In [ ]:
#@title 0) Setup Project Folders + (Optional) Mount Drive { display-mode: "form" }

use_google_drive = True #@param {type:"boolean"}
drive_base_path = "/content/gdrive/MyDrive" #@param {type:"string"}
drive_upload_folder = "dub_project" #@param {type:"string"}

import os
from pathlib import Path

if use_google_drive:
    from google.colab import drive
    drive.mount("/content/gdrive", force_remount=True)

PROJECT_ROOT = Path("/content/dub_project")
DIRS = {
    "input":    PROJECT_ROOT / "input",
    "work":     PROJECT_ROOT / "work",
    "extract":  PROJECT_ROOT / "work" / "00_extract",
    "output":   PROJECT_ROOT / "output",
    "tmp":      PROJECT_ROOT / "tmp",
    "logs":     PROJECT_ROOT / "logs",
}
for p in DIRS.values():
    p.mkdir(parents=True, exist_ok=True)

drive_upload_path = None
if use_google_drive:
    drive_upload_path = Path(drive_base_path) / drive_upload_folder
    drive_upload_path.mkdir(parents=True, exist_ok=True)
    print("📁 Drive folder ready:", drive_upload_path)

print("\n📂 Local project folders ready:")
for k, v in DIRS.items():
    print(f"  - {k}: {v}")

import subprocess
res = subprocess.run(["ffmpeg", "-version"], capture_output=True, text=True)
print("\n🎬 FFmpeg:", res.stdout.splitlines()[0] if res.returncode == 0 else "❌ Not found")

PIPELINE_PATHS = {}
print("\n✅ Step 0 done. Ready for Step 1.")

Mounted at /content/gdrive
📁 Drive folder ready: /content/gdrive/MyDrive/dub_project

📂 Local project folders ready:
  - input: /content/dub_project/input
  - work: /content/dub_project/work
  - extract: /content/dub_project/work/00_extract
  - output: /content/dub_project/output
  - tmp: /content/dub_project/tmp
  - logs: /content/dub_project/logs

🎬 FFmpeg: ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers

✅ Step 0 done. Ready for Step 1.


In [ ]:
#@title 1.0) Select video source + pick file_id { display-mode: "form" }

choose = "already_uploaded" #@param ["already_uploaded","upload_now","custom_path"]
custom_video_folder = "" #@param {type:"string"}

video_extensions = (".mp4", ".mov", ".mkv", ".avi", ".webm")

import os
import pandas as pd
from pathlib import Path

PROJECT_ROOT = globals().get("PROJECT_ROOT", Path("/content/dub_project"))
DIRS = globals().get("DIRS", {"input": PROJECT_ROOT / "input"})
use_google_drive = globals().get("use_google_drive", False)
drive_base_path = globals().get("drive_base_path", "/content/gdrive/MyDrive")
drive_upload_folder = globals().get("drive_upload_folder", "dub_project")

# تحديد مجلد الفيديو
if choose == "upload_now":
    from google.colab import files
    uploaded = files.upload()
    input_dir = Path(DIRS["input"])
    input_dir.mkdir(parents=True, exist_ok=True)
    for fn in uploaded.keys():
        src = Path("/content") / fn
        dst = input_dir / fn
        os.replace(str(src), str(dst))
    video_folder = str(input_dir)

elif choose == "custom_path":
    video_folder = (custom_video_folder or "").strip()
    if not video_folder:
        video_folder = str(DIRS["input"])

else:  # already_uploaded
    if use_google_drive and Path(drive_base_path).exists():
        video_folder = str(Path(drive_base_path) / drive_upload_folder)
    else:
        video_folder = str(DIRS["input"])

Path(video_folder).mkdir(parents=True, exist_ok=True)

ids, names = [], []
id_monitor = {}
video_id = 1

for f in sorted(os.listdir(video_folder)):
    if f.lower().endswith(video_extensions):
        ids.append(video_id)
        names.append(f)
        id_monitor[video_id] = f
        video_id += 1

df = pd.DataFrame({"file_name": names, "file_id": ids}).set_index("file_id")

print("📁 Folder:", video_folder)
if len(df):
    print("\n🎬 Available videos:")
    print(df)
else:
    print("⚠️  No videos found. Upload videos to:", video_folder)

📁 Folder: /content/gdrive/MyDrive/dub_project

🎬 Available videos:
                                                 file_name
file_id                                                   
1                             Cartoon Hero 2.0 Launch!.mp4
2        Learn English Romantic  Speak Fluently & Under...
3        There’s always room for dessert. #FRIENDS_720p...
4                                                video.mp4
5                                         video_dubbed.mp4
6                                video_no_audio_dubbed.mp4


In [3]:
#@title 1.1) Enter input File ID { display-mode: "form" }

file_id = 2 #@param {type:"number"}

import os

selected_name = id_monitor.get(int(file_id), None)

if selected_name is None:
    raise ValueError("❌ Invalid file_id. Choose a valid ID from the table above.")

INPUT_VIDEO = os.path.join(video_folder, selected_name)
print("✅ Selected video:", INPUT_VIDEO)
PIPELINE_PATHS["input_video"] = INPUT_VIDEO

✅ Selected video: /content/gdrive/MyDrive/dub_project/Learn English Romantic  Speak Fluently & Understand Real Conversations – FluentTalk Podcast - FluentTalk English Podcast (720p, h264).mp4


In [ ]:
#@title 1.2) Separate audio from video using FFmpeg { display-mode: "form" }

import subprocess
from pathlib import Path

def run_cmd(cmd):
    print("Running:\n", " ".join(map(str, cmd)))
    p = subprocess.run(cmd, capture_output=True, text=True)
    if p.returncode != 0:
        print("STDERR (tail):\n", (p.stderr or "")[-2000:])
        raise RuntimeError("Command failed.")
    return p

def has_audio_stream(video_path: Path) -> bool:
    cmd = [
        "ffprobe", "-v", "error",
        "-select_streams", "a:0",
        "-show_entries", "stream=codec_type",
        "-of", "default=nw=1:nk=1",
        str(video_path)
    ]
    p = subprocess.run(cmd, capture_output=True, text=True)
    return (p.returncode == 0) and ("audio" in (p.stdout or "").strip().lower())

def separate_audio_video(input_video_path: str, out_dir: str):
    input_video_path = Path(input_video_path)
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    if not input_video_path.exists():
        raise FileNotFoundError(f"Input video not found: {input_video_path}")

    base = input_video_path.stem
    video_no_audio = out_dir / f"{base}_no_audio.mp4"
    audio_raw_wav  = out_dir / f"{base}_raw.wav"
    audio_16k_mono = out_dir / f"{base}_16k_mono.wav"

    run_cmd([
        "ffmpeg", "-y",
        "-i", str(input_video_path),
        "-map", "0:v:0",
        "-c:v", "copy",
        "-an",
        str(video_no_audio)
    ])

    if not has_audio_stream(input_video_path):
        raise RuntimeError("❌ No audio stream found. Cannot continue ASR/dubbing pipeline.")

    run_cmd([
        "ffmpeg", "-y",
        "-i", str(input_video_path),
        "-map", "0:a:0",
        "-vn",
        "-acodec", "pcm_s16le",
        "-ar", "44100",
        "-ac", "2",
        str(audio_raw_wav)
    ])

    # ASR-friendly 16k mono wav
    run_cmd([
        "ffmpeg", "-y",
        "-i", str(audio_raw_wav),
        "-acodec", "pcm_s16le",
        "-ar", "16000",
        "-ac", "1",
        str(audio_16k_mono)
    ])

    return {
        "video_no_audio": str(video_no_audio),
        "audio_raw_wav": str(audio_raw_wav),
        "audio_16k_mono": str(audio_16k_mono),
    }

# Preconditions
if "DIRS" not in globals():
    raise NameError("DIRS not found. Run Step 0 first.")
if "INPUT_VIDEO" not in globals() or not INPUT_VIDEO:
    raise NameError("INPUT_VIDEO not found. Run Step 1.0/1.1 first.")

outputs = separate_audio_video(INPUT_VIDEO, str(DIRS["extract"]))

# حفظ المسارات
PIPELINE_PATHS.update(outputs)
PIPELINE_PATHS["extract_dir"] = str(DIRS["extract"])

print("\n✅ Step 1 outputs:")
for k, v in outputs.items():
    print(f"  - {k}: {v}")

Running:
 ffmpeg -y -i /content/gdrive/MyDrive/dub_project/Learn English Romantic  Speak Fluently & Understand Real Conversations – FluentTalk Podcast - FluentTalk English Podcast (720p, h264).mp4 -map 0:v:0 -c:v copy -an /content/dub_project/work/00_extract/Learn English Romantic  Speak Fluently & Understand Real Conversations – FluentTalk Podcast - FluentTalk English Podcast (720p, h264)_no_audio.mp4
Running:
 ffmpeg -y -i /content/gdrive/MyDrive/dub_project/Learn English Romantic  Speak Fluently & Understand Real Conversations – FluentTalk Podcast - FluentTalk English Podcast (720p, h264).mp4 -map 0:a:0 -vn -acodec pcm_s16le -ar 44100 -ac 2 /content/dub_project/work/00_extract/Learn English Romantic  Speak Fluently & Understand Real Conversations – FluentTalk Podcast - FluentTalk English Podcast (720p, h264)_raw.wav
Running:
 ffmpeg -y -i /content/dub_project/work/00_extract/Learn English Romantic  Speak Fluently & Understand Real Conversations – FluentTalk Podcast - FluentTalk Engl

In [5]:
#@title 2.0) Install Demucs + dependencies { display-mode: "form" }

import subprocess, sys

# Demucs
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "demucs"])

# Fix torchaudio save error (torchcodec)
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "torchcodec"])

# soundfile للـ workaround
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "soundfile"])

print("✅ Demucs + dependencies installed.")

✅ Demucs + dependencies installed.


In [ ]:
#@title 2.1) Demucs (vocals/background separation) { display-mode: "form" }

demucs_model = "htdemucs" #@param ["htdemucs","htdemucs_ft","mdx_extra","mdx_extra_q","mdx"]
device = "cuda" #@param ["cuda","cpu"]
segments_try = "4,2,1" #@param {type:"string"}
use_two_stems_vocals = True #@param {type:"boolean"}
shifts = 0 #@param {type:"integer"}
overlap = 0.25 #@param {type:"number"}

import os, sys, subprocess
from pathlib import Path
import torch

print("🔧 Preparing Demucs execution environment...")
subprocess.run([sys.executable, "-m", "pip", "install", "soundfile"], check=False)

patch_script_path = Path("demucs_runner.py")
patch_code = '''
import sys
import torch
import torchaudio
import soundfile

def safe_save(filepath, src, sample_rate, **kwargs):
    src = src.t().detach().cpu().numpy()
    subtype = None
    bps = kwargs.get("bits_per_sample", 16)
    if bps == 16:
        subtype = "PCM_16"
    elif bps == 24:
        subtype = "PCM_24"
    soundfile.write(filepath, src, sample_rate, subtype=subtype)

torchaudio.save = safe_save

from demucs.__main__ import main
if __name__ == "__main__":
    sys.exit(main())
'''
with open(patch_script_path, "w") as f:
    f.write(patch_code)
print(f"✅ Created patch script at {patch_script_path}")
# --------------------------------------------------------------------

if demucs_model.endswith("_q"):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "diffq"], check=False)

def run_cmd(cmd):
    print("Running:\n", " ".join(map(str, cmd)))
    p = subprocess.run(cmd, capture_output=True, text=True)
    if p.returncode != 0:
        print("\n--- STDOUT (tail) ---\n", (p.stdout or "")[-2000:])
        print("\n--- STDERR (tail) ---\n", (p.stderr or "")[-2000:])
        raise RuntimeError(f"Command failed (code={p.returncode}).")
    if p.stderr:
        print((p.stderr)[-600:])
    return p

# Preconditions
if "DIRS" not in globals():
    raise NameError("DIRS not found. Run Step 0 first.")
if "INPUT_VIDEO" not in globals() or not INPUT_VIDEO:
    raise NameError("INPUT_VIDEO not found. Run Step 1 first.")

input_video_path = Path(INPUT_VIDEO)
base = input_video_path.stem

# تحديد الصوت الخام
audio_raw = Path(PIPELINE_PATHS.get("audio_raw_wav", ""))
if not audio_raw.exists():
    candidates = sorted(Path(DIRS["extract"]).glob("*_raw.wav"))
    audio_raw = candidates[-1] if candidates else None

if audio_raw is None or not audio_raw.exists():
    raise FileNotFoundError("❌ Could not find extracted RAW audio. Run Step 1.2 first.")

print("🎵 Using RAW audio:", audio_raw)

demucs_out_root = Path(DIRS["work"]) / "01_demucs"
prep_root       = Path(DIRS["work"]) / "02_prep"
demucs_out_root.mkdir(parents=True, exist_ok=True)
prep_root.mkdir(parents=True, exist_ok=True)

# Parse segments
seg_list = []
for x in segments_try.split(","):
    x = x.strip()
    if x:
        seg_list.append(int(x))
if not seg_list:
    seg_list = [4, 2, 1]

def try_demucs(run_device: str):
    for seg in seg_list:
        cmd = [sys.executable, str(patch_script_path),
               "-n", demucs_model,
               "-d", run_device,
               "--jobs", "0",
               "--segment", str(seg),
               "--shifts", str(shifts),
               "--overlap", str(overlap),
               "-o", str(demucs_out_root),
               str(audio_raw)]

        if use_two_stems_vocals:
            cmd.insert(cmd.index("-o"), "--two-stems")
            cmd.insert(cmd.index("-o"), "vocals")

        print(f"\n🎯 Demucs attempt: device={run_device}, segment={seg}")
        try:
            run_cmd(cmd)
            return True
        except RuntimeError:
            print(f"⚠️  Failed with segment={seg} on {run_device}. Trying next...")
    return False

# Device fallback
real_device = device
if real_device == "cuda" and not torch.cuda.is_available():
    print("⚠️  CUDA not available. Falling back to CPU.")
    real_device = "cpu"

ok = try_demucs(real_device)
if not ok and real_device == "cuda":
    print("🔄 GPU failed. Falling back to CPU...")
    ok = try_demucs("cpu")
if not ok:
    raise RuntimeError("❌ Demucs failed on all settings.")

vocals_candidates = sorted(demucs_out_root.glob("**/vocals.wav"))
if not vocals_candidates:
    raise FileNotFoundError("❌ vocals.wav not found after Demucs.")
VOCALS_WAV = vocals_candidates[-1]

bg_candidates = []
bg_candidates += sorted(demucs_out_root.glob("**/no_vocals.wav"))
bg_candidates += sorted(demucs_out_root.glob("**/instrumental.wav"))
BKG_WAV = bg_candidates[-1] if bg_candidates else None

print("\n🎤 VOCALS_WAV:", VOCALS_WAV)
print("🎶 BKG_WAV   :", BKG_WAV if BKG_WAV else "Not found (optional)")

MODEL_INPUT_16K = prep_root / f"{base}_vocals_16k_mono.wav"
run_cmd([
    "ffmpeg", "-y",
    "-i", str(VOCALS_WAV),
    "-ac", "1",
    "-ar", "16000",
    "-acodec", "pcm_s16le",
    str(MODEL_INPUT_16K)
])

PIPELINE_PATHS["demucs_out_root"] = str(demucs_out_root)
PIPELINE_PATHS["vocals_wav"]      = str(VOCALS_WAV)
PIPELINE_PATHS["background_wav"]  = str(BKG_WAV) if BKG_WAV else None
PIPELINE_PATHS["model_input_16k"] = str(MODEL_INPUT_16K)

print("\n✅ Step 2.1 done.")

🔧 Preparing Demucs execution environment...
✅ Created patch script at demucs_runner.py
🎵 Using RAW audio: /content/dub_project/work/00_extract/Learn English Romantic  Speak Fluently & Understand Real Conversations – FluentTalk Podcast - FluentTalk English Podcast (720p, h264)_raw.wav

🎯 Demucs attempt: device=cuda, segment=4
Running:
 /usr/bin/python3 demucs_runner.py -n htdemucs -d cuda --jobs 0 --segment 4 --shifts 0 --overlap 0.25 --two-stems vocals -o /content/dub_project/work/01_demucs /content/dub_project/work/00_extract/Learn English Romantic  Speak Fluently & Understand Real Conversations – FluentTalk Podcast - FluentTalk English Podcast (720p, h264)_raw.wav
███████████████████████████████████████████████████████▌                  | 27.0/36.0 [00:03<00:00, 12.05seconds/s]
 83%|█████████████████████████████████████████████████████████████▋            | 30.0/36.0 [00:04<00:00, 12.65seconds/s]
 92%|███████████████████████████████████████████████████████████████████▊      | 33.0/36

In [ ]:
#@title 2.2) Speech Noise Separation Model { display-mode: "form" }

weights_filename = "/content/gdrive/MyDrive/dub_project/best_model.pth" #@param {type:"string"}
segment_seconds = 2.0 #@param {type:"number"}
overlap_ratio = 0.5 #@param {type:"number"}
base_channels = 32 #@param {type:"number"}

import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
import soundfile as sf
import numpy as np

# Preconditions
if "DIRS" not in globals():
    raise NameError("DIRS not found. Run Step 0 first.")
if "PIPELINE_PATHS" not in globals():
    raise NameError("PIPELINE_PATHS not found. Run Step 2.1 first.")

MODEL_INPUT_16K = Path(PIPELINE_PATHS.get("model_input_16k", ""))
if not MODEL_INPUT_16K.exists():
    raise FileNotFoundError(f"MODEL_INPUT_16K not found: {MODEL_INPUT_16K}")

device_t = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("🖥️  Device:", device_t)

WEIGHTS_PATH = Path(weights_filename.strip())
if not WEIGHTS_PATH.exists():
    raise FileNotFoundError(f"❌ Weights not found: {WEIGHTS_PATH}\n"
                            f"⚠️  ضع ملف best_model.pth في Drive أو غيّر المسار.")
print("📦 Weights:", WEIGHTS_PATH)

if int(base_channels) != 32:
    print("⚠️  base_channels should be 32. Forcing 32.")
    base_channels = 32

# ── Architecture ──────────────────────────────
def match_size(x, ref):
    _, _, H, W = x.shape
    _, _, Hr, Wr = ref.shape
    if H > Hr: x = x[:, :, :Hr, :]
    if W > Wr: x = x[:, :, :, :Wr]
    if x.shape[2] < Hr or x.shape[3] < Wr:
        pad_h = Hr - x.shape[2]
        pad_w = Wr - x.shape[3]
        x = F.pad(x, (0, pad_w, 0, pad_h))
    return x

class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.c1 = nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False)
        self.n1 = nn.InstanceNorm2d(out_ch, affine=True)
        self.c2 = nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False)
        self.n2 = nn.InstanceNorm2d(out_ch, affine=True)
        self.act = nn.LeakyReLU(0.2)
        self.skip = nn.Conv2d(in_ch, out_ch, 1, bias=False) if in_ch != out_ch else None

    def forward(self, x):
        i = x if self.skip is None else self.skip(x)
        x = self.act(self.n1(self.c1(x)))
        x = self.n2(self.c2(x))
        return self.act(x + i)

class Down(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, stride=2, padding=1, bias=False),
            nn.InstanceNorm2d(out_ch, affine=True),
            nn.LeakyReLU(0.2),
            ResBlock(out_ch, out_ch),
        )
    def forward(self, x):
        return self.net(x)

class Up(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_ch, out_ch, 4, 2, 1, bias=False)
        self.norm = nn.InstanceNorm2d(out_ch, affine=True)
        self.act = nn.LeakyReLU(0.2)
        self.res = ResBlock(out_ch + skip_ch, out_ch)

    def forward(self, x, skip):
        x = self.act(self.norm(self.up(x)))
        x = match_size(x, skip)
        x = torch.cat([x, skip], dim=1)
        return self.res(x)

class DualStreamResUNet(nn.Module):
    def __init__(self, base=32):
        super().__init__()
        self.enc1 = ResBlock(1, base)
        self.enc2 = Down(base, base*2)
        self.enc3 = Down(base*2, base*4)
        self.enc4 = Down(base*4, base*8)
        self.bot  = ResBlock(base*8, base*8)
        self.up3s = Up(base*8, base*4, base*4)
        self.up2s = Up(base*4, base*2, base*2)
        self.up1s = Up(base*2, base, base)
        self.up3n = Up(base*8, base*4, base*4)
        self.up2n = Up(base*4, base*2, base*2)
        self.up1n = Up(base*2, base, base)
        self.speech_head = nn.Sequential(nn.Conv2d(base, 1, 1), nn.Sigmoid())
        self.noise_head  = nn.Sequential(nn.Conv2d(base, 1, 1), nn.Sigmoid())

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(e1)
        e3 = self.enc3(e2)
        e4 = self.enc4(e3)
        b = self.bot(e4)
        xs = self.up3s(b, e3)
        xs = self.up2s(xs, e2)
        xs = self.up1s(xs, e1)
        xn = self.up3n(b, e3)
        xn = self.up2n(xn, e2)
        xn = self.up1n(xn, e1)
        return self.speech_head(xs), self.noise_head(xn)

def load_weights(model, path: Path):
    try:
        sd = torch.load(str(path), map_location="cpu", weights_only=True)
    except TypeError:
        sd = torch.load(str(path), map_location="cpu")
    if any(k.startswith("module.") for k in sd.keys()):
        sd = {k.replace("module.", ""): v for k, v in sd.items()}
    model.load_state_dict(sd, strict=True)
    return model

# ── STFT helpers ──────────────────────────────
CONFIG = {"SR": 16000, "SEG_SEC": float(segment_seconds), "N_FFT": 512, "HOP": 128, "WIN": 512}
STFT_WINDOW_CACHE = {}

def stft_mag_phase(wav: torch.Tensor):
    win_key = (wav.device.type, CONFIG["WIN"])
    if win_key not in STFT_WINDOW_CACHE:
        STFT_WINDOW_CACHE[win_key] = torch.hann_window(CONFIG["WIN"], device=wav.device)
    S = torch.stft(
        wav, n_fft=CONFIG["N_FFT"], hop_length=CONFIG["HOP"], win_length=CONFIG["WIN"],
        window=STFT_WINDOW_CACHE[win_key], return_complex=True
    )
    mag_log = torch.log1p(torch.abs(S))
    phase = torch.angle(S)
    return mag_log, phase

def istft_from_logmag_phase(mag_log: torch.Tensor, phase: torch.Tensor, length: int):
    win_key = (mag_log.device.type, CONFIG["WIN"])
    if win_key not in STFT_WINDOW_CACHE:
        STFT_WINDOW_CACHE[win_key] = torch.hann_window(CONFIG["WIN"], device=mag_log.device)
    mag = torch.expm1(mag_log).clamp_min(0.0)
    S = torch.polar(mag, phase)
    wav = torch.istft(
        S, n_fft=CONFIG["N_FFT"], hop_length=CONFIG["HOP"], win_length=CONFIG["WIN"],
        window=STFT_WINDOW_CACHE[win_key], length=length
    )
    return wav

@torch.no_grad()
def denoise_waveform(model, wav_1d: torch.Tensor):
    seg_len = int(CONFIG["SR"] * CONFIG["SEG_SEC"])
    hop = max(1, int(seg_len * (1.0 - float(overlap_ratio))))

    T0 = wav_1d.numel()
    wav = wav_1d
    if wav.numel() < seg_len:
        wav = F.pad(wav, (0, seg_len - wav.numel()))
    else:
        remainder = (wav.numel() - seg_len) % hop
        if remainder != 0:
            wav = F.pad(wav, (0, hop - remainder))

    T = wav.numel()
    win = torch.hann_window(seg_len, periodic=True)
    win = win / (win.max() + 1e-8)

    out = torch.zeros(T, dtype=torch.float32)
    wsum = torch.zeros(T, dtype=torch.float32)

    for start in range(0, T - seg_len + 1, hop):
        chunk = wav[start:start+seg_len].unsqueeze(0).to(device_t)
        mag_log, phase = stft_mag_phase(chunk)
        Ms, _ = model(mag_log.unsqueeze(1))
        mag_hat_log = torch.log1p(torch.expm1(mag_log) * Ms.squeeze(1))
        pred = istft_from_logmag_phase(mag_hat_log, phase, length=seg_len).squeeze(0).cpu()
        out[start:start+seg_len] += pred * win
        wsum[start:start+seg_len] += win

    out = out / (wsum + 1e-8)
    return out[:T0]

# ── Run ──────────────────────────────────────
model = DualStreamResUNet(base=int(base_channels)).to(device_t).eval()
model = load_weights(model, WEIGHTS_PATH)

print(f"📂 Loading {MODEL_INPUT_16K} via soundfile...")
wav_np, sr = sf.read(str(MODEL_INPUT_16K))
wav = torch.from_numpy(wav_np).float()

if wav.ndim == 2:
    wav = wav.mean(dim=1)  # mono mix

if sr != CONFIG["SR"]:
    import torchaudio.functional as taf
    wav = taf.resample(wav.unsqueeze(0), sr, CONFIG["SR"]).squeeze(0)

wav = wav.contiguous().float().cpu()

print("🧹 Denoising speech...")
clean = denoise_waveform(model, wav)

denoise_dir = Path(DIRS["work"]) / "02_denoise"
denoise_dir.mkdir(parents=True, exist_ok=True)

video_stem = Path(INPUT_VIDEO).stem
CLEAN_AUDIO_FOR_ASR = denoise_dir / f"{video_stem}_speech_clean_16k.wav"
sf.write(str(CLEAN_AUDIO_FOR_ASR), clean.numpy(), CONFIG["SR"], subtype="PCM_16")

print("✅ CLEAN_AUDIO_FOR_ASR =", CLEAN_AUDIO_FOR_ASR)

PIPELINE_PATHS["speech_clean_16k"] = str(CLEAN_AUDIO_FOR_ASR)

🖥️  Device: cuda
📦 Weights: /content/gdrive/MyDrive/dub_project/best_model.pth
📂 Loading /content/dub_project/work/02_prep/Learn English Romantic  Speak Fluently & Understand Real Conversations – FluentTalk Podcast - FluentTalk English Podcast (720p, h264)_vocals_16k_mono.wav via soundfile...
🧹 Denoising speech...
✅ CLEAN_AUDIO_FOR_ASR = /content/dub_project/work/02_denoise/Learn English Romantic  Speak Fluently & Understand Real Conversations – FluentTalk Podcast - FluentTalk English Podcast (720p, h264)_speech_clean_16k.wav


In [ ]:
#@title 3.0) Install WhisperX + pyannote + Gender Detection { display-mode: "form" }

import sys, subprocess, os

VENV = "/content/.venv"
PY = f"{VENV}/bin/python"

PYPI = "https://pypi.org/simple"
TORCH_CU128 = "https://download.pytorch.org/whl/cu128"
TORCH_CU121 = "https://download.pytorch.org/whl/cu121"
TORCH_CPU = "https://download.pytorch.org/whl/cpu"

os.environ["MPLBACKEND"] = "Agg"
os.environ["MPLCONFIGDIR"] = "/tmp/matplotlib-config"
os.makedirs("/tmp/matplotlib-config", exist_ok=True)

def run(cmd, check=True, tail=12000):
    print("\n>>", cmd)

    env = os.environ.copy()
    env["MPLBACKEND"] = "Agg"
    env["MPLCONFIGDIR"] = "/tmp/matplotlib-config"

    p = subprocess.run(
        cmd,
        shell=True,
        text=True,
        capture_output=True,
        env=env
    )

    if p.stdout:
        print("STDOUT tail:\n", p.stdout[-tail:])

    if p.stderr:
        print("STDERR tail:\n", p.stderr[-tail:])

    if check and p.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {p.returncode}:\n{cmd}")

    return p.returncode == 0

def verify_import(label, code):
    print(f"\n--- VERIFY: {label} ---")

    safe_code = (
        "import os\n"
        "os.environ['MPLBACKEND'] = 'Agg'\n"
        "os.environ['MPLCONFIGDIR'] = '/tmp/matplotlib-config'\n"
        "import traceback\n"
        "try:\n"
        + "\n".join("    " + line for line in code.strip().splitlines())
        + "\nexcept Exception:\n"
        f"    print('\\nFAILED: {label}')\n"
        "    traceback.print_exc()\n"
        "    raise\n"
    )

    return run(
        f'{PY} - << "PY"\n{safe_code}\nPY',
        check=True,
        tail=12000
    )

# System deps
run("apt-get -y -qq update && apt-get -y -qq install ffmpeg libsndfile1")
run("pip -q install -U uv")

# Clean isolated venv
run(f"rm -rf {VENV}")
run(f"uv venv {VENV} -p {sys.executable}")
run(f"uv pip install -p {PY} --default-index {PYPI} -U pip setuptools wheel")

ok = run(
    f"uv pip install -p {PY} --no-cache-dir "
    f"--index-url {TORCH_CU128} "
    "'torch==2.8.0' 'torchaudio==2.8.0' 'torchvision==0.23.0'",
    check=False
)

if not ok:
    print("\ncu128 failed, trying cu121...")
    ok = run(
        f"uv pip install -p {PY} --no-cache-dir "
        f"--index-url {TORCH_CU121} "
        "'torch==2.5.1' 'torchaudio==2.5.1' 'torchvision==0.20.1'",
        check=False
    )

if not ok:
    print("\nCUDA torch failed, trying CPU torch...")
    run(
        f"uv pip install -p {PY} --no-cache-dir "
        f"--index-url {TORCH_CPU} "
        "torch torchaudio torchvision"
    )

run(
    f"uv pip install -p {PY} --no-cache-dir "
    f"--default-index {PYPI} "
    "'tqdm>=4.67.1' "
    "'numpy>=2.0.2' "
    "soundfile pysrt pandas "
    "'transformers>=4.48,<5' "
    "'accelerate>=0.30,<1' "
    "matplotlib-inline"
)

# 3)  WhisperX
run(
    f"uv pip install -p {PY} --no-cache-dir "
    f"--default-index {PYPI} "
    f"--extra-index-url {TORCH_CU128} "
    "--index-strategy unsafe-best-match "
    "'whisperx==3.8.5'",
    tail=15000
)

# 4) Check deps
run(f"uv pip check -p {PY}", check=False, tail=12000)

# 5) Verify
verify_import(
    "torch",
    """
import torch
print("torch:", torch.__version__)
print("cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
"""
)

verify_import(
    "whisperx",
    """
import whisperx
print("whisperx: OK")
"""
)

verify_import(
    "diarization",
    """
from whisperx.diarize import DiarizationPipeline
print("DiarizationPipeline: OK")
"""
)

verify_import(
    "gender model libs",
    """
from transformers import Wav2Vec2ForSequenceClassification
print("Gender model lib: OK")
"""
)

print(f"\n✅ venv ready: {VENV}")


>> apt-get -y -qq update && apt-get -y -qq install ffmpeg libsndfile1
STDERR tail:
 W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


>> pip -q install -U uv
STDOUT tail:
    ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.7/24.7 MB 35.5 MB/s eta 0:00:00


>> rm -rf /content/.venv

>> uv venv /content/.venv -p /usr/bin/python3
STDERR tail:
Using CPython 3.12.13 interpreter at: /usr/bin/python3
Creating virtual environment at: .venv
Activate with: source .venv/bin/activate


>> uv pip install -p /content/.venv/bin/python --default-index https://pypi.org/simple -U pip setuptools wheel
STDERR tail:
 Resolved 4 packages in 147ms
Prepared 4 packages in 121ms
Installed 4 packages in 18ms
 + packaging==26.2
 + pip==26.1.1
 + setuptools==82.0.1
 + wheel==0.47.0


>> uv pip install -p /content/.venv/bin/python --no-cache-dir --index-url https://download.pytor

In [ ]:
#@title 3.1) HuggingFace Token (للـ Diarization) { display-mode: "form" }

import os, getpass


hf_token_param = "" #@param {type:"string"}

# 1) جرّب من .env
try:
    from dotenv import load_dotenv
    if os.path.exists("/content/.env"):
        load_dotenv("/content/.env", override=False)
except Exception:
    pass

HF_TOKEN = (hf_token_param or os.getenv("HF_TOKEN") or "").strip()

if not HF_TOKEN:
    print("⚠️  HF_TOKEN not found in env or param.")
    print("احصل على التوكن من: https://huggingface.co/settings/tokens")
    print("ثم وافق على:")
    print("  - https://huggingface.co/pyannote/speaker-diarization-3.1")
    print("  - https://huggingface.co/pyannote/segmentation-3.0")
    HF_TOKEN = getpass.getpass("Enter HF_TOKEN (hidden): ").strip()

if not HF_TOKEN:
    raise RuntimeError("❌ HF_TOKEN required for Diarization.")

os.environ["HF_TOKEN"] = HF_TOKEN
print("✅ HF_TOKEN loaded (hidden).")

✅ HF_TOKEN loaded (hidden).


In [10]:
#@title 3.2A) Write Integrated v5 Speaker Engine with Sentence Repair { display-mode: "form" }

from pathlib import Path

ENGINE_PATH = Path("/content/dubbing_v5_engine.py")

ENGINE_PATH.write_text(r'''
import os

os.environ["MPLBACKEND"] = "agg"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "true"

import argparse
import gc
import json
import time
import inspect
from pathlib import Path
from dataclasses import dataclass, asdict, field

import numpy as np
import torch
import torchaudio
import whisperx
from whisperx.diarize import DiarizationPipeline


@dataclass
class DubbingSegment:
    speaker: str
    gender: str
    start: float
    end: float
    text: str
    duration: float
    language: str
    words: list = field(default_factory=list)


def safe_empty_cache():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def clean_word_text(w):
    return str(w.get("word", "")).strip()


def force_binary_gender(value, default_gender="male"):
    value = str(value or "").strip().lower()

    if value == "female":
        return "female"

    if value == "male":
        return "male"

    return default_gender if default_gender in ("male", "female") else "male"


def normalize_gender_label(label):
    label = str(label or "").strip().lower()

    if "female" in label or label in ("f", "woman", "girl"):
        return "female"

    if "male" in label or label in ("m", "man", "boy"):
        return "male"

    return None


def load_whisperx_model_safe(model_name, device, kwargs):
    try:
        return whisperx.load_model(model_name, device, **kwargs)

    except TypeError as e:
        print(f"  ⚠️ WhisperX load_model failed with full kwargs: {e}")

        fallback = dict(kwargs)
        fallback.pop("vad_method", None)

        try:
            print("  ↪️ Retrying without vad_method...")
            return whisperx.load_model(model_name, device, **fallback)

        except TypeError as e2:
            print(f"  ⚠️ Retry without vad_method failed: {e2}")

        fallback.pop("asr_options", None)

        print("  ↪️ Retrying without vad_method/asr_options...")
        return whisperx.load_model(model_name, device, **fallback)


def create_diarization_pipeline(device, hf_token, diarization_model, fallback_model):
    diarize_kwargs_init = {"device": device}
    sig = inspect.signature(DiarizationPipeline)

    if "model_name" in sig.parameters:
        diarize_kwargs_init["model_name"] = diarization_model
    elif "model" in sig.parameters:
        diarize_kwargs_init["model"] = diarization_model
    else:
        print("  ⚠️ This WhisperX version does not expose a diarization model parameter. Using default model.")

    if "token" in sig.parameters:
        diarize_kwargs_init["token"] = hf_token
    elif "use_auth_token" in sig.parameters:
        diarize_kwargs_init["use_auth_token"] = hf_token
    else:
        raise RuntimeError(f"Unsupported DiarizationPipeline signature: {sig}")

    print(f"  🧠 Requested diarization model: {diarization_model}")

    try:
        return DiarizationPipeline(**diarize_kwargs_init), diarization_model

    except Exception as e:
        print(f"  ⚠️ Failed to load {diarization_model}: {e}")
        print(f"  ↪️ Falling back to {fallback_model}")

        fallback_kwargs = dict(diarize_kwargs_init)

        if "model_name" in fallback_kwargs:
            fallback_kwargs["model_name"] = fallback_model

        if "model" in fallback_kwargs:
            fallback_kwargs["model"] = fallback_model

        return DiarizationPipeline(**fallback_kwargs), fallback_model


def build_diarize_kwargs(mode, expected_speakers, min_speakers, max_speakers):
    mode = str(mode or "range").strip().lower()
    kwargs = {}

    if mode == "exact":
        n = int(max(1, expected_speakers))
        kwargs["num_speakers"] = n
        print(f"  🎯 Speaker count mode: exact | {n}")

    elif mode == "range":
        lo = int(max(1, min_speakers))
        hi = int(max(lo, max_speakers))

        kwargs["min_speakers"] = lo
        kwargs["max_speakers"] = hi

        print(f"  🎚️ Speaker count mode: range | {lo}-{hi}")

    else:
        print("  🤖 Speaker count mode: auto")

    return kwargs


def repair_missing_speakers(result):
    segments = result.get("segments", [])
    last_speaker = None

    for seg in segments:
        speaker = seg.get("speaker")

        if not speaker:
            word_speakers = [
                w.get("speaker")
                for w in seg.get("words", []) or []
                if w.get("speaker")
            ]

            if word_speakers:
                counts = {}

                for spk in word_speakers:
                    counts[spk] = counts.get(spk, 0) + 1

                speaker = max(counts, key=counts.get)

            elif last_speaker:
                speaker = last_speaker

            else:
                speaker = "SPEAKER_00"

            seg["speaker"] = speaker

        last_speaker = seg["speaker"]

        for w in seg.get("words", []) or []:
            if not w.get("speaker"):
                w["speaker"] = seg["speaker"]

    return result


def word_duration(w):
    if "start" not in w or "end" not in w:
        return 0.0

    return max(0.0, float(w["end"]) - float(w["start"]))


def is_sentence_end(word):
    text = clean_word_text(word)
    return text.endswith((".", "?", "!", "؟", "،", "؛", ":", ";"))


def split_words_into_sentence_units(words, max_gap=1.25):
    """
    Split word-level alignment into sentence-like units.
    This prevents short natural utterances from being split across speakers.
    """
    units = []
    cur = []
    last_end = None

    for w in words:
        if "start" not in w or "end" not in w or not clean_word_text(w):
            continue

        start = float(w["start"])
        end = float(w["end"])

        if cur and last_end is not None:
            gap = start - last_end

            if gap > max_gap:
                units.append(cur)
                cur = []

        cur.append(w)
        last_end = end

        if is_sentence_end(w):
            units.append(cur)
            cur = []
            last_end = None

    if cur:
        units.append(cur)

    return units


def speaker_duration_scores(words):
    scores = {}

    for w in words:
        spk = w.get("speaker")

        if not spk:
            continue

        dur = word_duration(w)

        # Avoid letting one badly aligned word dominate a whole sentence.
        dur = min(dur, 0.75)

        scores[spk] = scores.get(spk, 0.0) + max(0.05, dur)

    return scores


def speaker_word_counts(words):
    counts = {}

    for w in words:
        spk = w.get("speaker")

        if not spk:
            continue

        counts[spk] = counts.get(spk, 0) + 1

    return counts


def first_valid_speaker(words):
    for w in words:
        spk = w.get("speaker")

        if spk:
            return spk

    return None


def dominant_speaker_for_sentence(words, previous_speaker=None):
    """
    Pick the most stable speaker for a sentence-like unit.

    This fixes cases like:
    SPEAKER_00: Can I help
    SPEAKER_01: you?

    The intended result is:
    SPEAKER_00: Can I help you?
    """
    words = [
        w for w in words
        if "start" in w and "end" in w and clean_word_text(w)
    ]

    if not words:
        return previous_speaker or "SPEAKER_00"

    speakers = [w.get("speaker") for w in words if w.get("speaker")]
    unique_speakers = sorted(set(speakers))

    if len(unique_speakers) <= 1:
        return unique_speakers[0] if unique_speakers else (previous_speaker or "SPEAKER_00")

    counts = speaker_word_counts(words)
    scores = speaker_duration_scores(words)

    first_spk = first_valid_speaker(words)
    total_words = len(words)

    sent_start = float(words[0]["start"])
    sent_end = float(words[-1]["end"])
    sent_duration = max(0.0, sent_end - sent_start)

    # Strong anti-fragment rule.
    # A short sentence should not change speaker because of one final word.
    if total_words <= 7 and sent_duration <= 6.0:
        for spk in unique_speakers:
            if spk != first_spk and counts.get(spk, 0) <= 1:
                return first_spk

    # Specific common case:
    # "Can I help" / "you?"
    last_spk = words[-1].get("speaker")
    last_text = clean_word_text(words[-1])

    if (
        total_words <= 8
        and first_spk
        and last_spk != first_spk
        and counts.get(last_spk, 0) <= 1
        and last_text.endswith(("?", "؟", ".", "!"))
    ):
        return first_spk

    filtered_scores = dict(scores)

    # Do not let a one-word speaker fragment win inside a short sentence.
    for spk, cnt in counts.items():
        if cnt <= 1 and total_words <= 10:
            filtered_scores[spk] = filtered_scores.get(spk, 0.0) * 0.25

    if filtered_scores:
        return max(filtered_scores, key=filtered_scores.get)

    return first_spk or previous_speaker or "SPEAKER_00"


def repair_sentence_speaker_cohesion(result):
    """
    Repair speaker labels inside sentence-like units.

    Goal:
    - Prevent short utterances from being split across speakers.
    - Preserve real speaker changes between separate sentences.
    - Produce better dubbing segments.
    """
    previous_speaker = None

    for seg in result.get("segments", []):
        words = seg.get("words", []) or []

        timed_words = [
            w for w in words
            if "start" in w and "end" in w and clean_word_text(w)
        ]

        if not timed_words:
            if not seg.get("speaker"):
                seg["speaker"] = previous_speaker or "SPEAKER_00"

            previous_speaker = seg.get("speaker", previous_speaker)
            continue

        units = split_words_into_sentence_units(timed_words)

        for unit in units:
            chosen = dominant_speaker_for_sentence(
                unit,
                previous_speaker=previous_speaker,
            )

            for w in unit:
                w["speaker"] = chosen

            previous_speaker = chosen

        counts = speaker_word_counts(timed_words)

        if counts:
            seg["speaker"] = max(counts, key=counts.get)
            previous_speaker = seg["speaker"]
        else:
            seg["speaker"] = previous_speaker or "SPEAKER_00"

    return result


def transcribe_align_diarize(
    audio_path,
    language,
    whisper_model,
    batch_size,
    device,
    compute_type,
    hf_token,
    speaker_count_mode,
    expected_speakers,
    min_speakers,
    max_speakers,
    diarization_model,
    fallback_diarization_model,
):
    print("\n📝 [1/4] Transcription...")
    t0 = time.time()

    asr_options = {
        "initial_prompt": (
            "Educational or conversational video. Use clear punctuation. "
            "Preserve all spoken content and technical terms."
        ),
        "suppress_numerals": False,
    }

    kwargs = {
        "compute_type": compute_type,
        "asr_options": asr_options,
        "vad_method": "silero",
    }

    if language and language.strip():
        kwargs["language"] = language.strip()

    model = load_whisperx_model_safe(whisper_model, device, kwargs)
    audio = whisperx.load_audio(str(audio_path))

    result = model.transcribe(
        audio,
        batch_size=batch_size if device == "cuda" else max(1, min(4, batch_size)),
        chunk_size=15,
        print_progress=True,
    )

    detected_lang = result.get("language") or (
        language.strip() if language and language.strip() else None
    )

    if not detected_lang:
        raise RuntimeError("No language detected.")

    print(
        f"  ✅ ASR segments: {len(result.get('segments', []))} | "
        f"lang={detected_lang} | {time.time() - t0:.1f}s"
    )

    del model
    safe_empty_cache()

    print("\n🎯 [2/4] Alignment...")
    t0 = time.time()

    try:
        align_model, metadata = whisperx.load_align_model(
            language_code=detected_lang,
            device=device,
        )

        aligned = whisperx.align(
            result["segments"],
            align_model,
            metadata,
            audio,
            device,
            return_char_alignments=False,
            print_progress=False,
        )

        print(f"  ✅ Alignment done | {time.time() - t0:.1f}s")

        del align_model
        safe_empty_cache()

    except Exception as e:
        print(f"  ⚠️ Alignment failed: {e}")
        print("  ↪️ Continuing with ASR segment timestamps.")
        aligned = result

    print("\n🔊 [3/4] Diarization...")
    t0 = time.time()

    diarize_model, used_diarization_model = create_diarization_pipeline(
        device=device,
        hf_token=hf_token,
        diarization_model=diarization_model,
        fallback_model=fallback_diarization_model,
    )

    diarize_kwargs = build_diarize_kwargs(
        speaker_count_mode,
        expected_speakers,
        min_speakers,
        max_speakers,
    )

    diarize_segments = diarize_model(audio, **diarize_kwargs)

    print(f"  ✅ Diarization done | {time.time() - t0:.1f}s")

    print("\n🔗 [4/4] Assign speakers to words...")
    t0 = time.time()

    assigned = whisperx.assign_word_speakers(
        diarize_segments,
        aligned
    )

    assigned = repair_missing_speakers(assigned)
    assigned = repair_sentence_speaker_cohesion(assigned)

    speakers = sorted({
        seg.get("speaker")
        for seg in assigned.get("segments", [])
        if seg.get("speaker")
    })

    if not speakers:
        speakers = ["SPEAKER_00"]

        for seg in assigned.get("segments", []):
            seg["speaker"] = "SPEAKER_00"

            for w in seg.get("words", []) or []:
                w["speaker"] = "SPEAKER_00"

    print(
        f"  ✅ Speakers: {len(speakers)} | "
        f"segments: {len(assigned.get('segments', []))} | "
        f"{time.time() - t0:.1f}s"
    )

    return {
        "segments": assigned.get("segments", []),
        "language": detected_lang,
        "speakers": speakers,
        "diarization_model": used_diarization_model,
    }


def estimate_pitch_gender(audio_chunk, sample_rate, default_gender="male"):
    try:
        if len(audio_chunk) < int(sample_rate * 0.8):
            return default_gender, 0.50, "default_too_short_for_pitch"

        tensor = torch.tensor(audio_chunk, dtype=torch.float32).unsqueeze(0)

        pitch = torchaudio.functional.detect_pitch_frequency(
            tensor,
            sample_rate=sample_rate,
        ).squeeze().detach().cpu().numpy()

        pitch = pitch[np.isfinite(pitch)]
        pitch = pitch[(pitch > 60) & (pitch < 400)]

        if len(pitch) < 5:
            return default_gender, 0.50, "default_no_stable_pitch"

        median_f0 = float(np.median(pitch))

        if median_f0 >= 165:
            return "female", 0.58, f"pitch_fallback_f0_{median_f0:.1f}"

        return "male", 0.58, f"pitch_fallback_f0_{median_f0:.1f}"

    except Exception as e:
        return default_gender, 0.50, f"default_pitch_failed_{type(e).__name__}"


def detect_genders(
    audio_path,
    segments,
    speakers,
    device,
    conf_threshold=0.62,
    min_audio_sec=1.5,
    max_audio_sec_per_speaker=35.0,
    default_gender="male",
):
    from transformers import Wav2Vec2FeatureExtractor, Wav2Vec2ForSequenceClassification

    print("\n🧬 Voice gender detection...")

    model_name = "prithivMLmods/Common-Voice-Gender-Detection"

    processor = Wav2Vec2FeatureExtractor.from_pretrained(model_name)
    model = Wav2Vec2ForSequenceClassification.from_pretrained(model_name).to(device)
    model.eval()

    id2label = model.config.id2label

    waveform, sample_rate = torchaudio.load(str(audio_path))

    if waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0, keepdim=True)

    if sample_rate != 16000:
        waveform = torchaudio.transforms.Resample(
            orig_freq=sample_rate,
            new_freq=16000,
        )(waveform)
        sample_rate = 16000

    waveform_np = waveform.squeeze().detach().cpu().numpy()

    speaker_chunks = {s: [] for s in speakers}

    for seg in segments:
        spk = seg.get("speaker")

        if not spk or spk not in speaker_chunks:
            continue

        start = float(seg.get("start", 0.0))
        end = float(seg.get("end", 0.0))
        dur = max(0.0, end - start)

        if dur >= 0.6:
            speaker_chunks[spk].append({
                "start": start,
                "end": end,
                "duration": dur,
            })

    for spk in speaker_chunks:
        speaker_chunks[spk].sort(key=lambda x: x["duration"], reverse=True)

        selected = []
        total = 0.0

        for c in speaker_chunks[spk]:
            if total >= max_audio_sec_per_speaker:
                break

            selected.append(c)
            total += c["duration"]

        speaker_chunks[spk] = selected

    gender_map = {}
    gender_conf = {}
    gender_evidence = {}

    for spk in sorted(speakers):
        weighted = {
            "male": 0.0,
            "female": 0.0,
        }

        votes = []
        pitch_pool = []
        total_used_sec = 0.0
        sample_count = 0

        male_score = None
        female_score = None

        for c in speaker_chunks.get(spk, []):
            start_i = max(0, int(c["start"] * sample_rate))
            end_i = min(len(waveform_np), int(c["end"] * sample_rate))

            if end_i <= start_i:
                continue

            chunk = waveform_np[start_i:end_i]
            chunk_sec = len(chunk) / float(sample_rate)

            if chunk_sec < 0.6:
                continue

            peak = float(np.max(np.abs(chunk))) + 1e-8
            chunk = chunk / peak

            pitch_pool.append(chunk)

            inputs = processor(
                chunk,
                sampling_rate=16000,
                return_tensors="pt",
                padding=True,
            )

            inputs = {k: v.to(device) for k, v in inputs.items()}

            with torch.no_grad():
                logits = model(**inputs).logits
                probs = torch.softmax(logits, dim=1)[0]
                pred = int(torch.argmax(probs).item())
                conf = float(probs[pred].item())
                label = normalize_gender_label(id2label.get(pred, pred))

            if label in weighted:
                weighted[label] += conf * chunk_sec
                total_used_sec += chunk_sec
                sample_count += 1

                votes.append({
                    "label": label,
                    "confidence": round(conf, 4),
                    "duration": round(chunk_sec, 3),
                })

        source = "classifier"
        reason = "ok"

        if total_used_sec >= min_audio_sec and sample_count > 0:
            total_score = weighted["male"] + weighted["female"] + 1e-8
            male_score = float(weighted["male"] / total_score)
            female_score = float(weighted["female"] / total_score)

            gender = "male" if male_score >= female_score else "female"
            score = max(male_score, female_score)

            if score < conf_threshold:
                reason = "low_confidence_forced_best_score"

            final_gender = force_binary_gender(gender, default_gender)
            final_conf = round(score, 4)

        else:
            if pitch_pool:
                merged = np.concatenate(pitch_pool)

                final_gender, final_conf, pitch_reason = estimate_pitch_gender(
                    merged,
                    sample_rate,
                    default_gender=default_gender,
                )

                source = "pitch_fallback"
                reason = pitch_reason

            else:
                final_gender = force_binary_gender(default_gender, "male")
                final_conf = 0.50
                source = "default_fallback"
                reason = "no_audio_evidence"

        final_gender = force_binary_gender(final_gender, default_gender)

        gender_map[spk] = final_gender
        gender_conf[spk] = final_conf
        gender_evidence[spk] = {
            "source": source,
            "reason": reason,
            "used_audio_sec": round(float(total_used_sec), 3),
            "samples": int(sample_count),
            "male_score": round(float(male_score), 4) if male_score is not None else None,
            "female_score": round(float(female_score), 4) if female_score is not None else None,
            "votes": votes,
        }

        print(
            f"  {spk}: {final_gender} | "
            f"confidence={final_conf:.0%} | "
            f"source={source} | "
            f"audio={total_used_sec:.1f}s"
        )

    del model, processor
    safe_empty_cache()

    return gender_map, gender_conf, gender_evidence


def assign_default_genders(speakers, default_gender="male"):
    gender_map = {}
    gender_conf = {}
    gender_evidence = {}

    for spk in sorted(speakers):
        gender = force_binary_gender(default_gender, "male")

        gender_map[spk] = gender
        gender_conf[spk] = 0.50
        gender_evidence[spk] = {
            "source": "gender_detection_disabled",
            "reason": "default_binary_voice_gender",
            "used_audio_sec": 0.0,
            "samples": 0,
            "votes": [],
        }

    return gender_map, gender_conf, gender_evidence


def build_dubbing_segments(
    raw,
    gender_map,
    min_duration=0.25,
    max_duration=5.5,
    max_chars=95,
    default_gender="male",
):
    output = []
    language = raw["language"]

    def gender_for(spk):
        return force_binary_gender(gender_map.get(spk), default_gender)

    def flush(words, speaker):
        if not words:
            return

        speaker = speaker or "SPEAKER_00"

        valid = [
            w for w in words
            if "start" in w and "end" in w and clean_word_text(w)
        ]

        if not valid:
            return

        start = float(valid[0]["start"])
        end = float(valid[-1]["end"])
        duration = round(end - start, 3)

        if duration < min_duration:
            return

        text = " ".join(clean_word_text(w) for w in valid).strip()

        if not text:
            return

        output.append(DubbingSegment(
            speaker=speaker,
            gender=gender_for(speaker),
            start=round(start, 3),
            end=round(end, 3),
            text=text,
            duration=duration,
            language=language,
            words=[
                {
                    "word": clean_word_text(w),
                    "start": round(float(w["start"]), 3),
                    "end": round(float(w["end"]), 3),
                    "speaker": w.get("speaker", speaker),
                }
                for w in valid
            ],
        ))

    for seg in raw.get("segments", []):
        speaker = seg.get("speaker", "SPEAKER_00")
        words = seg.get("words", []) or []

        timed_words = [
            w for w in words
            if "start" in w and "end" in w and clean_word_text(w)
        ]

        if timed_words:
            chunk = []
            chunk_start = None
            cur_speaker = None

            for w in timed_words:
                word = clean_word_text(w)

                if not word:
                    continue

                spk = w.get("speaker", speaker) or speaker
                w["speaker"] = spk

                if cur_speaker is not None and spk != cur_speaker:
                    chunk_text_so_far = " ".join(clean_word_text(x) for x in chunk).strip()
                    current_word_text = clean_word_text(w)

                    protect_short_sentence = (
                        len(chunk) <= 7
                        and not chunk_text_so_far.endswith((".", "?", "!", "؟"))
                        and current_word_text.endswith(("?", "؟", ".", "!"))
                    )

                    if protect_short_sentence:
                        w["speaker"] = cur_speaker
                        spk = cur_speaker
                    else:
                        flush(chunk, cur_speaker)
                        chunk = []
                        chunk_start = None

                if chunk_start is None:
                    chunk_start = float(w["start"])

                chunk.append(w)
                cur_speaker = spk

                chunk_text = " ".join(clean_word_text(x) for x in chunk).strip()
                chunk_dur = float(w["end"]) - chunk_start
                ends_sentence = word.endswith((".", "?", "!", "؟", "،", "؛", ":", ";"))

                if (
                    chunk_dur >= max_duration
                    or len(chunk_text) >= max_chars
                    or (ends_sentence and chunk_dur >= 1.2)
                ):
                    flush(chunk, cur_speaker)
                    chunk = []
                    chunk_start = None
                    cur_speaker = None

            if chunk:
                flush(chunk, cur_speaker or speaker)

        else:
            text = str(seg.get("text", "")).strip()
            start = float(seg.get("start", 0.0))
            end = float(seg.get("end", start))
            duration = round(end - start, 3)

            if not text or duration < min_duration:
                continue

            output.append(DubbingSegment(
                speaker=speaker,
                gender=gender_for(speaker),
                start=round(start, 3),
                end=round(end, 3),
                text=text,
                duration=duration,
                language=language,
                words=[],
            ))

    output.sort(key=lambda x: (x.start, x.end))

    print(f"\n🎬 Dubbing segments built: {len(output)}")

    return output


def build_speaker_stats(segments, speakers, gender_map, gender_conf, gender_evidence, default_gender="male"):
    stats = {}

    for spk in sorted(speakers):
        spk_segments = [s for s in segments if s.speaker == spk]
        total = round(sum(float(s.duration) for s in spk_segments), 3)

        stats[spk] = {
            "gender": force_binary_gender(gender_map.get(spk), default_gender),
            "gender_confidence": gender_conf.get(spk, 0.50),
            "segments": len(spk_segments),
            "total_speech_sec": total,
            "avg_segment_sec": round(total / max(len(spk_segments), 1), 3),
            "gender_evidence": gender_evidence.get(spk, {}),
        }

    return stats


def write_srt(segments, srt_path):
    def fmt(seconds):
        seconds = max(0.0, float(seconds))
        ms = int(round((seconds % 1) * 1000))

        if ms >= 1000:
            seconds += 1.0
            ms = 0

        s = int(seconds) % 60
        m = (int(seconds) // 60) % 60
        h = int(seconds) // 3600

        return f"{h:02d}:{m:02d}:{s:02d},{ms:03d}"

    with open(srt_path, "w", encoding="utf-8") as f:
        for i, seg in enumerate(segments, 1):
            f.write(f"{i}\n")
            f.write(f"{fmt(seg.start)} --> {fmt(seg.end)}\n")
            f.write(f"{seg.text}\n\n")


def main():
    ap = argparse.ArgumentParser()

    ap.add_argument("--audio", required=True)
    ap.add_argument("--out_json", required=True)
    ap.add_argument("--out_srt", required=True)
    ap.add_argument("--out_debug_json", required=True)

    ap.add_argument("--model", default="large-v3")
    ap.add_argument("--language", default="")
    ap.add_argument("--device", default="cuda")
    ap.add_argument("--compute_type", default="float16")
    ap.add_argument("--batch_size", type=int, default=16)

    ap.add_argument("--speaker_count_mode", default="exact", choices=["auto", "exact", "range"])
    ap.add_argument("--expected_speakers", type=int, default=2)
    ap.add_argument("--min_speakers", type=int, default=1)
    ap.add_argument("--max_speakers", type=int, default=3)

    ap.add_argument("--diarization_model", default="pyannote/speaker-diarization-community-1")
    ap.add_argument("--fallback_diarization_model", default="pyannote/speaker-diarization-3.1")

    ap.add_argument("--detect_gender", action="store_true")
    ap.add_argument("--gender_conf_threshold", type=float, default=0.62)
    ap.add_argument("--min_gender_audio_sec", type=float, default=1.5)
    ap.add_argument("--max_gender_audio_sec_per_speaker", type=float, default=35.0)
    ap.add_argument("--default_fallback_gender", default="male", choices=["male", "female"])

    ap.add_argument("--max_segment_duration", type=float, default=5.5)
    ap.add_argument("--max_segment_chars", type=int, default=95)
    ap.add_argument("--min_segment_duration", type=float, default=0.25)

    ap.add_argument("--hf_token", required=True)

    args = ap.parse_args()

    audio_path = Path(args.audio)
    out_json = Path(args.out_json)
    out_srt = Path(args.out_srt)
    out_debug_json = Path(args.out_debug_json)

    out_json.parent.mkdir(parents=True, exist_ok=True)
    out_srt.parent.mkdir(parents=True, exist_ok=True)
    out_debug_json.parent.mkdir(parents=True, exist_ok=True)

    if not audio_path.exists():
        raise FileNotFoundError(f"Audio not found: {audio_path}")

    device = "cuda" if args.device == "cuda" and torch.cuda.is_available() else "cpu"
    compute_type = args.compute_type

    if device == "cpu" and compute_type.lower() in ("float16", "fp16"):
        compute_type = "int8"

    default_gender = force_binary_gender(args.default_fallback_gender, "male")

    print(f"🖥️ Device: {device} | compute_type: {compute_type}")

    raw = transcribe_align_diarize(
        audio_path=audio_path,
        language=args.language,
        whisper_model=args.model,
        batch_size=args.batch_size,
        device=device,
        compute_type=compute_type,
        hf_token=args.hf_token,
        speaker_count_mode=args.speaker_count_mode,
        expected_speakers=args.expected_speakers,
        min_speakers=args.min_speakers,
        max_speakers=args.max_speakers,
        diarization_model=args.diarization_model,
        fallback_diarization_model=args.fallback_diarization_model,
    )

    speakers = raw["speakers"]

    if args.detect_gender and speakers:
        gender_map, gender_conf, gender_evidence = detect_genders(
            audio_path=audio_path,
            segments=raw["segments"],
            speakers=speakers,
            device=device,
            conf_threshold=args.gender_conf_threshold,
            min_audio_sec=args.min_gender_audio_sec,
            max_audio_sec_per_speaker=args.max_gender_audio_sec_per_speaker,
            default_gender=default_gender,
        )
    else:
        gender_map, gender_conf, gender_evidence = assign_default_genders(
            speakers,
            default_gender=default_gender,
        )

    gender_map = {
        spk: force_binary_gender(gender_map.get(spk), default_gender)
        for spk in speakers
    }

    segments = build_dubbing_segments(
        raw,
        gender_map,
        min_duration=args.min_segment_duration,
        max_duration=args.max_segment_duration,
        max_chars=args.max_segment_chars,
        default_gender=default_gender,
    )

    speaker_stats = build_speaker_stats(
        segments=segments,
        speakers=speakers,
        gender_map=gender_map,
        gender_conf=gender_conf,
        gender_evidence=gender_evidence,
        default_gender=default_gender,
    )

    payload = {
        "segments": [asdict(s) for s in segments],
        "language": raw["language"],

        "speakers": speakers,
        "detected_speaker_count": len(speakers),

        "speaker_gender": gender_map,
        "speaker_gender_confidence": gender_conf,
        "speaker_gender_evidence": gender_evidence,
        "speaker_stats": speaker_stats,

        "speaker_count_mode": args.speaker_count_mode,
        "expected_speakers": args.expected_speakers,
        "min_speakers": args.min_speakers,
        "max_speakers": args.max_speakers,

        "diarization_model": raw.get("diarization_model"),
        "whisper_model": args.model,
        "gender_model": "prithivMLmods/Common-Voice-Gender-Detection" if args.detect_gender else None,
        "source": "integrated_v5_engine_with_sentence_repair",

        "dubbing_policy": {
            "binary_gender_only": True,
            "never_emit_unknown_gender": True,
            "default_fallback_gender": default_gender,
            "sentence_speaker_cohesion_repair": True,
        },
    }

    debug = {
        "raw_segments_after_sentence_repair": raw.get("segments", []),
        "speaker_stats": speaker_stats,
        "speaker_gender": gender_map,
        "speaker_gender_confidence": gender_conf,
        "speaker_gender_evidence": gender_evidence,
    }

    with open(out_json, "w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)

    with open(out_debug_json, "w", encoding="utf-8") as f:
        json.dump(debug, f, ensure_ascii=False, indent=2)

    write_srt(segments, out_srt)

    print(f"\n💾 JSON saved: {out_json}")
    print(f"💾 SRT saved: {out_srt}")
    print(f"💾 Debug JSON saved: {out_debug_json}")

    print(f"\n{'=' * 60}")
    print("  📊 Final summary")
    print(f"{'=' * 60}")
    print(f"  Speakers: {len(speakers)}")
    print(f"  Segments: {len(segments)}")
    print(f"  Language: {raw['language']}")
    print(f"  Diarization model: {raw.get('diarization_model')}")

    for spk in speakers:
        st = speaker_stats.get(spk, {})
        ev = st.get("gender_evidence", {})

        print(
            f"  {spk}: {st.get('gender', default_gender)} | "
            f"confidence={st.get('gender_confidence', 0.50):.0%} | "
            f"source={ev.get('source', 'n/a')} | "
            f"segments={st.get('segments', 0)} | "
            f"speech={st.get('total_speech_sec', 0.0):.1f}s"
        )


if __name__ == "__main__":
    main()
''', encoding="utf-8")

print(f"✅ Engine written: {ENGINE_PATH}")

✅ Engine written: /content/dubbing_v5_engine.py


In [11]:
#@title 3.2B) Run Integrated v5 Diarization for Dubbing { display-mode: "form" }

# User-facing settings
language_code = "en" #@param {type:"string"}
speaker_count_mode = "exact" #@param ["exact", "range", "auto"]
expected_speakers = 2 #@param {type:"integer"}
min_speakers = 1 #@param {type:"integer"}
max_speakers = 3 #@param {type:"integer"}
detect_gender = True #@param {type:"boolean"}

from pathlib import Path
import os, json, subprocess

# Internal defaults
WHISPER_MODEL_NAME = "large-v3"
DEVICE_PREFERENCE = "cuda"
COMPUTE_TYPE_PREF = "float16"
BATCH_SIZE = 16

DIARIZATION_MODEL_NAME = "pyannote/speaker-diarization-community-1"
FALLBACK_DIARIZATION_MODEL_NAME = "pyannote/speaker-diarization-3.1"

GENDER_CONF_THRESHOLD = 0.62
MIN_GENDER_AUDIO_SEC = 1.5
MAX_GENDER_AUDIO_SEC_PER_SPEAKER = 35.0
DEFAULT_FALLBACK_GENDER = "male"

MAX_SEGMENT_DURATION = 5.5
MAX_SEGMENT_CHARS = 95
MIN_SEGMENT_DURATION = 0.25

os.environ["MPLBACKEND"] = "agg"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "true"

if "DIRS" not in globals():
    raise NameError("DIRS not found. Run Step 0 first.")

if "PIPELINE_PATHS" not in globals():
    raise NameError("PIPELINE_PATHS not found. Run earlier steps.")

if "INPUT_VIDEO" not in globals() or not INPUT_VIDEO:
    raise NameError("INPUT_VIDEO not found. Run Step 1 first.")

if "HF_TOKEN" not in os.environ or not os.environ.get("HF_TOKEN", "").strip():
    raise RuntimeError("HF_TOKEN is missing. Run the Hugging Face token cell first.")

if "ENGINE_PATH" not in globals():
    ENGINE_PATH = Path("/content/dubbing_v5_engine.py")

if not ENGINE_PATH.exists():
    raise FileNotFoundError("Engine file not found. Run cell 3.2A first.")

CLEAN_AUDIO = Path(PIPELINE_PATHS.get("speech_clean_16k", ""))

if not CLEAN_AUDIO.exists():
    raise FileNotFoundError("speech_clean_16k was not found. Run Step 2.2 first.")

asr_dir = Path(DIRS["work"]) / "03_asr_diarize"
asr_dir.mkdir(parents=True, exist_ok=True)

video_stem = Path(INPUT_VIDEO).stem

OUT_JSON = asr_dir / f"{video_stem}_diarization.json"
OUT_SRT = asr_dir / f"{video_stem}_original_en.srt"
OUT_DEBUG_JSON = asr_dir / f"{video_stem}_diarization_debug.json"

for p in [OUT_JSON, OUT_SRT, OUT_DEBUG_JSON]:
    if p.exists():
        p.unlink()

hf_token = os.environ.get("HF_TOKEN", "").strip()

cmd = [
    "/content/.venv/bin/python",
    str(ENGINE_PATH),

    "--audio", str(CLEAN_AUDIO),
    "--out_json", str(OUT_JSON),
    "--out_srt", str(OUT_SRT),
    "--out_debug_json", str(OUT_DEBUG_JSON),

    "--model", WHISPER_MODEL_NAME,
    "--language", (language_code or ""),
    "--device", DEVICE_PREFERENCE,
    "--compute_type", COMPUTE_TYPE_PREF,
    "--batch_size", str(BATCH_SIZE),

    "--speaker_count_mode", speaker_count_mode,
    "--expected_speakers", str(expected_speakers),
    "--min_speakers", str(min_speakers),
    "--max_speakers", str(max_speakers),

    "--diarization_model", DIARIZATION_MODEL_NAME,
    "--fallback_diarization_model", FALLBACK_DIARIZATION_MODEL_NAME,

    "--gender_conf_threshold", str(GENDER_CONF_THRESHOLD),
    "--min_gender_audio_sec", str(MIN_GENDER_AUDIO_SEC),
    "--max_gender_audio_sec_per_speaker", str(MAX_GENDER_AUDIO_SEC_PER_SPEAKER),
    "--default_fallback_gender", DEFAULT_FALLBACK_GENDER,

    "--max_segment_duration", str(MAX_SEGMENT_DURATION),
    "--max_segment_chars", str(MAX_SEGMENT_CHARS),
    "--min_segment_duration", str(MIN_SEGMENT_DURATION),

    "--hf_token", hf_token,
]

if detect_gender:
    cmd.append("--detect_gender")

env = os.environ.copy()
env["MPLBACKEND"] = "agg"
env["TOKENIZERS_PARALLELISM"] = "false"
env["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "true"

p = subprocess.run(
    cmd,
    env=env,
    capture_output=True,
    text=True,
)

print(p.stdout)

if p.returncode != 0:
    print("ERROR (stderr tail):\n", (p.stderr or "")[-10000:])
    raise RuntimeError(f"Integrated v5 diarization failed (code={p.returncode})")

if not OUT_JSON.exists():
    raise FileNotFoundError(f"JSON not found: {OUT_JSON}")

if not OUT_SRT.exists():
    raise FileNotFoundError(f"SRT not found: {OUT_SRT}")

with open(OUT_JSON, "r", encoding="utf-8") as f:
    diarization_data = json.load(f)

PIPELINE_PATHS["asr_srt_en"] = str(OUT_SRT)
PIPELINE_PATHS["diarization_json"] = str(OUT_JSON)
PIPELINE_PATHS["speaker_debug_json"] = str(OUT_DEBUG_JSON)

PIPELINE_PATHS["speaker_gender_map"] = diarization_data.get("speaker_gender", {})
PIPELINE_PATHS["speaker_gender_confidence"] = diarization_data.get("speaker_gender_confidence", {})
PIPELINE_PATHS["speaker_gender_evidence"] = diarization_data.get("speaker_gender_evidence", {})
PIPELINE_PATHS["speaker_stats"] = diarization_data.get("speaker_stats", {})

PIPELINE_PATHS["speakers_list"] = diarization_data.get("speakers", [])
PIPELINE_PATHS["detected_speaker_count"] = diarization_data.get("detected_speaker_count", 0)
PIPELINE_PATHS["detected_language"] = diarization_data.get("language", "en")
PIPELINE_PATHS["speaker_count_mode"] = diarization_data.get("speaker_count_mode", speaker_count_mode)
PIPELINE_PATHS["diarization_model"] = diarization_data.get("diarization_model", DIARIZATION_MODEL_NAME)

print("\n✅ Integrated v5 diarization complete:")
print(f"  - SRT        : {OUT_SRT}")
print(f"  - JSON       : {OUT_JSON}")
print(f"  - Debug JSON : {OUT_DEBUG_JSON}")
print(f"  - Speakers   : {PIPELINE_PATHS['speakers_list']}")
print(f"  - Count      : {PIPELINE_PATHS['detected_speaker_count']}")
print(f"  - Gender map : {PIPELINE_PATHS['speaker_gender_map']}")

srt_text = OUT_SRT.read_text(encoding="utf-8")
srt_blocks = [b for b in srt_text.strip().split("\n\n") if b.strip()]

print(f"\n📝 SRT segments count: {len(srt_blocks)}")
print("\nPreview:")
print("\n\n".join(srt_blocks[:8]))

print("\n📊 Speaker stats:")
for spk, stats in PIPELINE_PATHS["speaker_stats"].items():
    print(f"  {spk}: {stats}")

🖥️ Device: cuda | compute_type: float16

📝 [1/4] Transcription...
2026-05-09 13:09:59 - whisperx.vads.silero - INFO - Performing voice activity detection using Silero...
Downloading: "https://github.com/snakers4/silero-vad/zipball/master" to /root/.cache/torch/hub/master.zip
Progress: 33.33%...
Progress: 66.67%...
Progress: 100.00%...
  ✅ ASR segments: 3 | lang=en | 49.1s

🎯 [2/4] Alignment...
Downloading: "https://download.pytorch.org/torchaudio/models/wav2vec2_fairseq_base_ls960_asr_ls960.pth" to /root/.cache/torch/hub/checkpoints/wav2vec2_fairseq_base_ls960_asr_ls960.pth
  ✅ Alignment done | 5.9s

🔊 [3/4] Diarization...
  🧠 Requested diarization model: pyannote/speaker-diarization-community-1
2026-05-09 13:10:13 - whisperx.diarize - INFO - Loading diarization model: pyannote/speaker-diarization-community-1
  🎯 Speaker count mode: exact | 2
  ✅ Diarization done | 5.2s

🔗 [4/4] Assign speakers to words...
  ✅ Speakers: 2 | segments: 14 | 0.0s

🧬 Voice gender detection...
  SPEAKER_00:

In [13]:
#@title 3.3) Display Diarization Summary { display-mode: "form" }

import json
import pandas as pd
from pathlib import Path

if "PIPELINE_PATHS" not in globals() or "diarization_json" not in PIPELINE_PATHS:
    raise NameError("Run Step 3.2 first.")

with open(PIPELINE_PATHS["diarization_json"], "r", encoding="utf-8") as f:
    data = json.load(f)

segments = data["segments"]
speakers = data["speakers"]
gender_map = data["speaker_gender"]

# جدول المقاطع
ICON = {"male": "👨", "female": "👩", "unknown": "❓"}
df = pd.DataFrame([{
    "#":        i + 1,
    "Speaker":  s["speaker"],
    "Gender":   f"{ICON.get(s['gender'], '❓')} {s['gender']}",
    "Start":    f"{s['start']:.1f}s",
    "End":      f"{s['end']:.1f}s",
    "Duration": f"{s['duration']:.1f}s",
    "Text":     s["text"][:70] + ("..." if len(s["text"]) > 70 else ""),
} for i, s in enumerate(segments)])

print(f"📊 Total: {len(segments)} segments | {len(speakers)} speakers\n")
print(df.to_string(index=False))

print(f"\n👥 Speakers breakdown:")
for spk in speakers:
    g = gender_map.get(spk, "unknown")
    spk_segs = [s for s in segments if s["speaker"] == spk]
    total_dur = sum(s["duration"] for s in spk_segs)
    icon = ICON.get(g, "❓")
    print(f"  {icon} {spk}: {g} | {len(spk_segs)} segments | {total_dur:.1f}s ({total_dur/60:.1f} min)")

📊 Total: 13 segments | 2 speakers

 #    Speaker   Gender Start   End Duration                                           Text
 1 SPEAKER_00   👨 male  0.3s  4.5s     4.2s                                Can I help you?
 2 SPEAKER_01 👩 female  4.5s  5.5s     1.0s                                Yes, thank you.
 3 SPEAKER_01 👩 female  6.1s  7.0s     0.9s                     I'm going to get that one.
 4 SPEAKER_00   👨 male  9.5s 10.1s     0.6s                                       Begonia.
 5 SPEAKER_00   👨 male 10.2s 10.9s     0.6s                                   Nice choice.
 6 SPEAKER_01 👩 female 11.3s 12.0s     0.7s                            Is that what it is?
 7 SPEAKER_01 👩 female 15.1s 15.6s     0.5s                                  It is pretty.
 8 SPEAKER_01 👩 female 16.5s 18.1s     1.5s                                          Yeah.
 9 SPEAKER_00   👨 male 21.7s 23.5s     1.8s             Don't let any pets eat the leaves.
10 SPEAKER_00   👨 male 23.6s 24.6s     1.1s            

In [14]:
#@title 4.0) Translation Provider Settings (Gemini/Groq) {display-mode: "form"}

import os, sys, subprocess
from pathlib import Path

PROVIDER = "gemini"  #@param ["gemini","groq"]
DOMAIN   = "general"  #@param ["general","technical","medical","academic"]

GEMINI_MODEL = "gemini-2.5-flash"  #@param {type:"string"}

GROQ_MODEL = "qwen/qwen3-32b"  #@param ["openai/gpt-oss-120b","qwen/qwen3-32b","custom"]
GROQ_MODEL_CUSTOM = ""  #@param {type:"string"}
if GROQ_MODEL == "custom":
    GROQ_MODEL = (GROQ_MODEL_CUSTOM or "").strip() or "qwen/qwen3-32b"

# تثبيت المتطلبات
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U",
                       "python-dotenv", "groq", "google-genai", "google-generativeai",
                       "pysrt", "pydub", "edge-tts", "aiohttp", "nest_asyncio", "soundfile"])

from dotenv import load_dotenv
DOTENV_PATH = "/content/.env"
p = Path(DOTENV_PATH)
if p.exists():
    load_dotenv(dotenv_path=str(p), override=False)
    print(f"✅ Loaded .env from {DOTENV_PATH}")
else:
    print(f"⚠️  .env not found at {DOTENV_PATH}. أنشئه أو أدخل المفتاح يدوياً.")

import getpass

def require_env(name: str, prompt: str = None) -> str:
    v = os.environ.get(name, "").strip()
    if not v and prompt:
        v = getpass.getpass(prompt).strip()
        os.environ[name] = v
    if not v:
        raise RuntimeError(f"❌ Missing {name}")
    return v

print(f"\n📌 Provider: {PROVIDER} | Domain: {DOMAIN}")

if PROVIDER == "groq":
    require_env("GROQ_API_KEY", "Enter GROQ_API_KEY (hidden): ")
    print(f"✅ GROQ_API_KEY loaded | Model: {GROQ_MODEL}")
else:
    require_env("GEMINI_API_KEY", "Enter GEMINI_API_KEY (hidden): ")
    print(f"✅ GEMINI_API_KEY loaded | Model: {GEMINI_MODEL}")

⚠️  .env not found at /content/.env. أنشئه أو أدخل المفتاح يدوياً.

📌 Provider: gemini | Domain: general
Enter GEMINI_API_KEY (hidden): ··········
✅ GEMINI_API_KEY loaded | Model: gemini-2.5-flash


In [15]:
#@title 4.1) Arabic Translation (Time-aware + Auto-shorten) {display-mode:"form"}

# ──────── Tunables ────────
AR_CHARS_PER_SEC = 13.0   #@param {type:"number"}
MIN_MAX_CHARS    = 12     #@param {type:"integer"}
SOFT_OVER_BUDGET = 1.15   #@param {type:"number"}
MAX_SPEEDUP_NEED = 1.8    #@param {type:"number"}
MAX_COMPRESS_ROUNDS = 2   #@param {type:"integer"}
MAX_CHARS_PER_CHUNK = 4500  #@param {type:"integer"}
MAX_RETRIES = 6           #@param {type:"integer"}

import os, re, json, time
from pathlib import Path
import pysrt

# Preconditions
if "PIPELINE_PATHS" not in globals():
    raise NameError("Run earlier steps first.")
if "DIRS" not in globals() or "INPUT_VIDEO" not in globals():
    raise NameError("DIRS / INPUT_VIDEO missing.")

SRC_SRT = Path(PIPELINE_PATHS.get("asr_srt_en", ""))
if not SRC_SRT.exists():
    raise FileNotFoundError(f"❌ Missing asr_srt_en: {SRC_SRT}. Run Step 3.2 first.")

translate_dir = DIRS["work"] / "04_translate"
translate_dir.mkdir(parents=True, exist_ok=True)

video_stem = Path(INPUT_VIDEO).stem
OUT_SRT = translate_dir / f"{video_stem}_ar_dub.srt"

# Provider config
PROVIDER     = globals().get("PROVIDER", "groq")
DOMAIN       = globals().get("DOMAIN", "general")
GEMINI_MODEL = globals().get("GEMINI_MODEL", "gemini-2.0-flash-exp")
GROQ_MODEL   = globals().get("GROQ_MODEL", "qwen/qwen3-32b")

# ──────── Helpers ────────
def extract_json_array(text: str):
    t = (text or "").replace("```json", "").replace("```", "").strip()
    lb, rb = t.find("["), t.rfind("]")
    if lb != -1 and rb != -1 and rb > lb:
        try:
            return json.loads(t[lb:rb+1].strip())
        except Exception:
            pass
    m = re.search(r"\[\s*\{.*\}\s*\]", t, flags=re.DOTALL)
    if not m:
        raise ValueError("No JSON array found in LLM output.")
    return json.loads(m.group(0))

def call_groq(prompt: str) -> str:
    from groq import Groq
    key = os.getenv("GROQ_API_KEY", "").strip()
    if not key:
        raise RuntimeError("Missing GROQ_API_KEY")
    client = Groq(api_key=key)
    base = dict(model=str(GROQ_MODEL),
                messages=[{"role": "user", "content": prompt}],
                temperature=0.25, top_p=1, stream=False)
    try:
        r = client.chat.completions.create(**base, max_completion_tokens=6000)
    except TypeError:
        r = client.chat.completions.create(**base, max_tokens=6000)
    return r.choices[0].message.content

def call_gemini(prompt: str) -> str:
    from google import genai
    from google.genai import types
    key = os.getenv("GEMINI_API_KEY", "").strip()
    if not key:
        raise RuntimeError("Missing GEMINI_API_KEY")
    client = genai.Client(api_key=key)
    r = client.models.generate_content(
        model=str(GEMINI_MODEL), contents=prompt,
        config=types.GenerateContentConfig(temperature=0.25),
    )
    return r.text or ""

def llm(prompt: str) -> str:
    return call_gemini(prompt) if PROVIDER == "gemini" else call_groq(prompt)

def chunk_items(items, max_chars=4500):
    chunks, cur, cur_len = [], [], 0
    for it in items:
        line = f'{it["i"]}: dur={it["dur"]}s max={it["max_chars"]} | {it["text"]}'
        if cur and (cur_len + len(line) + 1 > max_chars):
            chunks.append(cur); cur=[]; cur_len=0
        cur.append(it); cur_len += len(line) + 1
    if cur: chunks.append(cur)
    return chunks

def build_prompt_timeaware(chunk):
    lines = "\n".join([f'{x["i"]}: dur={x["dur"]}s max={x["max_chars"]} | {x["text"]}' for x in chunk])
    return f"""
أنت محرّر نصوص دبلجة محترف.

المهمة:
أعد صياغة كل سطر إلى العربية الفصحى الحديثة (MSA) بصياغة منطوقة مناسبة للدبلجة، مع الالتزام بزمن السطر.

قواعد صارمة:
- حافظ على المعنى دون إضافة أو حذف معلومات.
- لا تستخدم العامية أو لهجات محكية.
- التزم قدر الإمكان بحد "max" لعدد الأحرف في كل سطر.
- حافظ على الأسماء والمصطلحات التقنية والأرقام بدقة.
- لا تغيّر الأرقام داخل أسماء الإصدارات/الموديلات أو الرموز التقنية (مثل GPT-4, v2.1, USB 3.0).
- اترك الأرقام كما هي (سيتم التعامل معها لاحقاً إذا لزم).
- أخرج فقط JSON array بالشكل:
  [{{"i":..,"t":".."}}, ...]
- كل "i" يجب أن يطابق نفس رقم السطر وبنفس الترتيب.

المجال: {DOMAIN}

المدخل:
{lines}
""".strip()

def build_prompt_compress(chunk):
    lines = "\n".join([f'{x["i"]}: max={x["max_chars"]} | EN: {x["en"]} | AR: {x["ar"]}' for x in chunk])
    return f"""
أنت محرّر دبلجة محترف.

المهمة:
قصّر فقط النص العربي ليلائم حد "max" دون خسارة المعنى أو التفاصيل المهمة.

قواعد صارمة:
- لا تضف معلومات جديدة ولا تحذف نقاطًا أساسية.
- أبقِ الأسماء والمصطلحات التقنية دقيقة.
- العربية الفصحى الحديثة فقط (بدون عامية).
- اترك الأرقام كما هي.
- أخرج فقط JSON array:
  [{{"i":..,"t":".."}}, ...]
- كل "i" يجب أن يطابق نفس رقم السطر وبنفس الترتيب.

المجال: {DOMAIN}

المدخل:
{lines}
""".strip()

def translate_chunk_timeaware(chunk):
    prompt = build_prompt_timeaware(chunk)
    expected = [x["i"] for x in chunk]
    last_err = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            out = llm(prompt)
            arr = extract_json_array(out)
            got = [int(x["i"]) for x in arr]
            if len(arr) != len(chunk) or got != expected:
                raise ValueError("Index/count mismatch")
            return {int(x["i"]): str(x["t"]).strip() for x in arr}
        except Exception as e:
            last_err = e
            time.sleep(min(20, 2 ** attempt))
    raise RuntimeError(f"Translation failed: {last_err}")

def compress_chunk(chunk):
    prompt = build_prompt_compress(chunk)
    expected = [x["i"] for x in chunk]
    last_err = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            out = llm(prompt)
            arr = extract_json_array(out)
            got = [int(x["i"]) for x in arr]
            if len(arr) != len(chunk) or got != expected:
                raise ValueError("Index/count mismatch")
            return {int(x["i"]): str(x["t"]).strip() for x in arr}
        except Exception as e:
            last_err = e
            time.sleep(min(20, 2 ** attempt))
    raise RuntimeError(f"Compression failed: {last_err}")

# ──────── Load SRT ────────
subs = pysrt.open(str(SRC_SRT), encoding="utf-8")

items = []
for s in subs:
    en = re.sub(r"\s*\n\s*", " ", (s.text or "").strip()).strip()
    dur_sec = max(0.20, (s.end.ordinal - s.start.ordinal) / 1000.0)
    max_chars = max(MIN_MAX_CHARS, int(dur_sec * AR_CHARS_PER_SEC))
    items.append({"i": int(s.index), "text": en, "dur": round(dur_sec, 2), "max_chars": int(max_chars)})

chunks = chunk_items(items, MAX_CHARS_PER_CHUNK)
print(f"📝 Lines={len(items)} | Chunks={len(chunks)} | Provider={PROVIDER} | Domain={DOMAIN}")

# Pass 1
mapping = {}
for ci, ch in enumerate(chunks, start=1):
    mapping.update(translate_chunk_timeaware(ch))
    print(f"  ✅ Pass1 chunk {ci}/{len(chunks)} done.")

# Risk detection
def est_need(ar_text: str, max_chars: int):
    L = max(1, len((ar_text or "").strip()))
    return L / max(1, int(max_chars))

def find_bad(mapping_dict):
    bad = []
    for it in items:
        i = it["i"]
        ar = (mapping_dict.get(i, "") or "").strip()
        need = est_need(ar, it["max_chars"])
        if need > SOFT_OVER_BUDGET or need > MAX_SPEEDUP_NEED:
            bad.append({"i": i, "en": it["text"], "ar": ar, "max_chars": it["max_chars"], "need": round(need, 2)})
    bad.sort(key=lambda x: x["need"], reverse=True)
    return bad

def chunk_bad(bad, max_chars=4500):
    chunks, cur, cur_len = [], [], 0
    for x in bad:
        line = f'{x["i"]}: max={x["max_chars"]} | EN: {x["en"]} | AR: {x["ar"]}'
        if cur and (cur_len + len(line) + 1 > max_chars):
            chunks.append(cur); cur=[]; cur_len=0
        cur.append(x); cur_len += len(line) + 1
    if cur: chunks.append(cur)
    return chunks

# Pass 2 (compress risky only)
for r in range(1, MAX_COMPRESS_ROUNDS + 1):
    bad = find_bad(mapping)
    print(f"  🔍 Compression round {r}/{MAX_COMPRESS_ROUNDS} | risky={len(bad)}")
    if not bad:
        break
    for ci, ch in enumerate(chunk_bad(bad, MAX_CHARS_PER_CHUNK), start=1):
        fixed = compress_chunk(ch)
        mapping.update(fixed)
        print(f"    ✅ Compress chunk {ci}/{len(chunk_bad(bad, MAX_CHARS_PER_CHUNK))} done.")

# Save
for s in subs:
    s.text = mapping.get(int(s.index), (s.text or "").strip())

subs.save(str(OUT_SRT), encoding="utf-8")
PIPELINE_PATHS["srt_ar_dub"] = str(OUT_SRT)

print(f"\n✅ Saved: {OUT_SRT}")

📝 Lines=13 | Chunks=1 | Provider=gemini | Domain=general
  ✅ Pass1 chunk 1/1 done.
  🔍 Compression round 1/2 | risky=2
    ✅ Compress chunk 1/1 done.
  🔍 Compression round 2/2 | risky=1
    ✅ Compress chunk 1/1 done.

✅ Saved: /content/dub_project/work/04_translate/Learn English Romantic  Speak Fluently & Understand Real Conversations – FluentTalk Podcast - FluentTalk English Podcast (720p, h264)_ar_dub.srt


In [ ]:
#@title 4.2) Convert numbers to Arabic words (for TTS) {display-mode:"form"}

import re
from pathlib import Path
import pysrt

in_path = Path(PIPELINE_PATHS.get("srt_ar_dub", ""))
if not in_path.exists():
    raise FileNotFoundError(f"Missing srt_ar_dub: {in_path}")

out_path = in_path.with_name(in_path.stem + "_tts_numbers.srt")

DIGIT_TRANS = str.maketrans("٠١٢٣٤٥٦٧٨٩۰۱۲۳۴۵۶۷۸۹", "01234567890123456789")

ONES = ["صفر","واحد","اثنان","ثلاثة","أربعة","خمسة","ستة","سبعة","ثمانية","تسعة"]
TENS = ["","عشرة","عشرون","ثلاثون","أربعون","خمسون","ستون","سبعون","ثمانون","تسعون"]
TEENS = {11:"أحد عشر",12:"اثنا عشر",13:"ثلاثة عشر",14:"أربعة عشر",15:"خمسة عشر",
         16:"ستة عشر",17:"سبعة عشر",18:"ثمانية عشر",19:"تسعة عشر"}
HUNDREDS = {1:"مئة",2:"مئتان",3:"ثلاثمئة",4:"أربعمئة",5:"خمسمئة",
            6:"ستمئة",7:"سبعمئة",8:"ثمانمئة",9:"تسعمئة"}

SCALES = [
    ("","","",""),
    ("ألف","ألفان","آلاف","ألف"),
    ("مليون","مليونان","ملايين","مليون"),
    ("مليار","ملياران","مليارات","مليار"),
    ("تريليون","تريليونان","تريليونات","تريليون"),
]

def two_digits(n: int) -> str:
    if n < 10: return ONES[n]
    if n == 10: return "عشرة"
    if 11 <= n <= 19: return TEENS[n]
    t, u = divmod(n, 10)
    if u == 0: return TENS[t]
    return f"{ONES[u]} و{TENS[t]}"

def three_digits(n: int) -> str:
    if n < 100: return two_digits(n)
    h, r = divmod(n, 100)
    htxt = HUNDREDS[h]
    return htxt if r == 0 else f"{htxt} و{two_digits(r)}"

def int_to_words(n: int) -> str:
    if n == 0: return "صفر"
    parts = []
    group_idx = 0
    while n > 0:
        n, group = divmod(n, 1000)
        if group == 0:
            group_idx += 1; continue
        group_words = three_digits(group)
        if group_idx >= len(SCALES):
            sg, dl, pl, mn = SCALES[-1]
        else:
            sg, dl, pl, mn = SCALES[group_idx]
        if group_idx == 0:
            parts.append(group_words)
        else:
            if group == 1: parts.append(sg)
            elif group == 2: parts.append(dl)
            elif 3 <= group <= 10: parts.append(f"{group_words} {pl}")
            else: parts.append(f"{group_words} {mn}")
        group_idx += 1
    return " و".join(reversed(parts))

def decimal_to_words(num_str: str) -> str:
    s = num_str.replace(",", "").strip()
    sign = ""
    if s.startswith("+"): s = s[1:]
    elif s.startswith("-"): sign = "سالب "; s = s[1:]
    if "." in s:
        ip, fp = s.split(".", 1)
        ip = ip if ip else "0"
        fp = fp.rstrip("0")
        base = int_to_words(int(ip))
        if not fp: return sign + base
        frac = " ".join(ONES[int(d)] for d in fp if d.isdigit())
        return sign + f"{base} فاصل {frac}"
    return sign + int_to_words(int(s))

def should_skip_number(text: str, start: int, end: int) -> bool:
    """تجاهل الأرقام داخل tokens تقنية إنجليزية (GPT-4, v2.1, USB3.0)"""
    prev = text[start-1] if start-1 >= 0 else ""
    nxt  = text[end] if end < len(text) else ""
    token = text[start:end]
    if re.match(r"[A-Za-z]", prev) or re.match(r"[A-Za-z]", nxt):
        return True
    if start-2 >= 0:
        prev2 = text[start-2]
        if re.match(r"[A-Za-z]", prev2) and prev in "-_/":
            return True
    pre = text[max(0, start-12):start]
    if "." in token and re.search(r"[A-Za-z]{2,10}\s*$", pre):
        return True
    return False

def normalize_separators(s: str) -> str:
    return (s or "").replace("٬", ",").replace("٫", ".").replace("،", ",")

num_re = re.compile(r"[+-]?\d[\d,]*([.]\d+)?%?")

def convert_numbers_in_text(s: str) -> str:
    if not s: return s
    s2 = s.translate(DIGIT_TRANS)
    s2 = normalize_separators(s2)
    def repl(m):
        raw = m.group(0)
        st, en = m.start(), m.end()
        if should_skip_number(s2, st, en): return raw
        is_pct = raw.endswith("%")
        core = raw[:-1] if is_pct else raw
        try:
            w = decimal_to_words(core)
            return (w + " بالمئة") if is_pct else w
        except Exception:
            return raw
    return num_re.sub(repl, s2)

subs = pysrt.open(str(in_path), encoding="utf-8")
for sub in subs:
    lines = sub.text.splitlines()
    sub.text = "\n".join(convert_numbers_in_text(ln) for ln in lines)

subs.save(str(out_path), encoding="utf-8")
PIPELINE_PATHS["srt_ar_dub_tts_numbers"] = str(out_path)

print(f"✅ TTS-ready SRT saved: {out_path}")

✅ TTS-ready SRT saved: /content/dub_project/work/04_translate/Learn English Romantic  Speak Fluently & Understand Real Conversations – FluentTalk Podcast - FluentTalk English Podcast (720p, h264)_ar_dub_tts_numbers.srt


In [28]:
#@title 5.0) TTS Provider + Speaker Voice Mapping - ElevenLabs + XTTS Fallback {display-mode:"form"}

import os, sys, subprocess, getpass, json, re, time, hashlib
from pathlib import Path

# ──────── Provider ────────
# elevenlabs_then_xtts:
#   1. Try ElevenLabs first.
#   2. If ElevenLabs fails or is blocked, use local XTTS voice cloning.
TTS_PROVIDER = "elevenlabs_then_xtts"  #@param ["elevenlabs_then_xtts","xtts","elevenlabs"]

# ──────── ElevenLabs ────────
ELEVEN_MODEL_ID = "eleven_multilingual_v2"  #@param {type:"string"}
ELEVEN_VOICE_ID_MALE   = "pCKbQ4EPGE06zpEPGNvS"  #@param {type:"string"}
ELEVEN_VOICE_ID_FEMALE = "VwC51uc4PUblWEJSPzeo"  #@param {type:"string"}
ELEVEN_OUTPUT_FORMAT = "mp3_44100_128"  #@param ["mp3_44100_128","mp3_44100_192"]

# ──────── XTTS ────────
# XTTS will automatically clone each speaker voice from the original video audio.
XTTS_LANGUAGE = "ar"  #@param {type:"string"}
XTTS_MAX_REF_SEC_PER_SPEAKER = 14.0  #@param {type:"number"}
XTTS_MIN_REF_SEC_PER_SPEAKER = 2.0   #@param {type:"number"}

# Optional fallback reference voices.
# If a speaker does not have enough clean reference audio, these are used.
XTTS_MALE_FALLBACK_REF = "/content/voices/male_ref.wav"      #@param {type:"string"}
XTTS_FEMALE_FALLBACK_REF = "/content/voices/female_ref.wav"  #@param {type:"string"}

# In dubbing, final voice gender must always be male/female.
DEFAULT_FALLBACK_GENDER = "male"  #@param ["male","female"]

# ──────── Base dependencies ────────
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "-U",
    "python-dotenv",
    "soundfile"
])

from dotenv import load_dotenv

DOTENV_PATH = "/content/.env"
if Path(DOTENV_PATH).exists():
    load_dotenv(dotenv_path=str(DOTENV_PATH), override=False)


def require_env_optional(name: str, prompt: str = None) -> str:
    value = os.environ.get(name, "").strip()

    if not value and prompt:
        try:
            value = getpass.getpass(prompt).strip()
            if value:
                os.environ[name] = value
        except Exception:
            value = ""

    return value


def force_binary_gender(gender, default=DEFAULT_FALLBACK_GENDER):
    gender = str(gender or "").strip().lower()

    if gender == "female":
        return "female"

    if gender == "male":
        return "male"

    return default if default in ("male", "female") else "male"


# ──────── ElevenLabs setup ────────
USE_ELEVEN = TTS_PROVIDER in ("elevenlabs", "elevenlabs_then_xtts")

if USE_ELEVEN:
    if not ELEVEN_VOICE_ID_MALE.strip() or not ELEVEN_VOICE_ID_FEMALE.strip():
        raise RuntimeError("ElevenLabs requires both MALE and FEMALE voice IDs.")

    eleven_key = require_env_optional(
        "ELEVEN_API_KEY",
        "Enter ELEVEN_API_KEY, leave empty to use XTTS fallback: "
    )

    if eleven_key:
        print("✅ ELEVEN_API_KEY loaded.")
    else:
        if TTS_PROVIDER == "elevenlabs":
            raise RuntimeError("TTS_PROVIDER is elevenlabs, but ELEVEN_API_KEY is missing.")

        print("⚠️ ELEVEN_API_KEY not provided. Will use XTTS fallback.")


# ──────── XTTS setup ────────
USE_XTTS = TTS_PROVIDER in ("xtts", "elevenlabs_then_xtts")

if USE_XTTS:
    try:
        from TTS.api import TTS
    except Exception:
        print("Installing XTTS / Coqui TTS for Python 3.12. This may take a few minutes...")

        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", "-U",
            "coqui-tts",
            "soundfile"
        ])

        from TTS.api import TTS

    import torch

    XTTS_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

    print(f"Loading XTTS v2 on {XTTS_DEVICE}...")
    xtts_model = TTS("tts_models/multilingual/multi-dataset/xtts_v2").to(XTTS_DEVICE)
    print("✅ XTTS v2 loaded.")


# ──────── Build voice map ────────
GENDER_VOICE_MAP = {
    "elevenlabs": {
        "male": ELEVEN_VOICE_ID_MALE,
        "female": ELEVEN_VOICE_ID_FEMALE,
    },
    "xtts": {
        "male": "XTTS_AUTO_SPEAKER_REF",
        "female": "XTTS_AUTO_SPEAKER_REF",
    },
    "elevenlabs_then_xtts": {
        "male": ELEVEN_VOICE_ID_MALE,
        "female": ELEVEN_VOICE_ID_FEMALE,
    },
}

voices = GENDER_VOICE_MAP[TTS_PROVIDER]

print(f"\n🔊 TTS Provider: {TTS_PROVIDER}")
print("\n🎙️ Voice mapping:")
print(f"  👨 male   → {voices['male']}")
print(f"  👩 female → {voices['female']}")


# ──────── Show speaker → voice assignment ────────
if "PIPELINE_PATHS" in globals() and "speaker_gender_map" in PIPELINE_PATHS:
    gender_map = {
        spk: force_binary_gender(gender)
        for spk, gender in PIPELINE_PATHS["speaker_gender_map"].items()
    }

    print("\n👥 Speaker → Voice assignment:")

    for spk, gender in gender_map.items():
        voice = voices.get(gender, voices["male"])
        icon = {"male": "👨", "female": "👩"}.get(gender, "👨")
        print(f"  {icon} {spk} ({gender}) → {voice}")
else:
    print("\n⚠️ Speaker gender map not found. Run Step 3 first.")

✅ ELEVEN_API_KEY loaded.
Installing XTTS / Coqui TTS for Python 3.12. This may take a few minutes...
Loading XTTS v2 on cuda...
 > You must confirm the following:
 | > "I have purchased a commercial license from Coqui: licensing@coqui.ai"
 | > "Otherwise, I agree to the terms of the non-commercial CPML: https://coqui.ai/cpml" - [y/n]
 | | > y


100%|██████████| 1.87G/1.87G [00:26<00:00, 71.3MiB/s]
4.37kiB [00:00, 3.11MiB/s]
361kiB [00:00, 27.5MiB/s]
100%|██████████| 32.0/32.0 [00:00<00:00, 72.4kiB/s]
100%|██████████| 7.75M/7.75M [00:00<00:00, 58.4MiB/s]


✅ XTTS v2 loaded.

🔊 TTS Provider: elevenlabs_then_xtts

🎙️ Voice mapping:
  👨 male   → pCKbQ4EPGE06zpEPGNvS
  👩 female → VwC51uc4PUblWEJSPzeo

👥 Speaker → Voice assignment:
  👨 SPEAKER_00 (male) → pCKbQ4EPGE06zpEPGNvS
  👩 SPEAKER_01 (female) → VwC51uc4PUblWEJSPzeo


In [29]:
#@title 5.1) Build Dub Vocals - ElevenLabs + XTTS Voice Cloning Fallback {display-mode:"form"}

# ──────── Settings ────────
max_speedup   = 1.8   #@param {type:"number"}
vocal_gain_db = 3.0   #@param {type:"number"}
target_sr     = 48000 #@param {type:"integer"}
fade_ms       = 8     #@param {type:"integer"}

import os, re, sys, asyncio, subprocess, time, hashlib, json, shutil
from pathlib import Path

import nest_asyncio
nest_asyncio.apply()

import pysrt
from pydub import AudioSegment

# ──────── Preconditions ────────
if "DIRS" not in globals() or "PIPELINE_PATHS" not in globals():
    raise NameError("Run earlier steps first.")

TTS_PROVIDER = globals().get("TTS_PROVIDER", "xtts")
GENDER_VOICE_MAP = globals().get("GENDER_VOICE_MAP", {})

ELEVEN_MODEL_ID = globals().get("ELEVEN_MODEL_ID", "eleven_multilingual_v2")
ELEVEN_OUTPUT_FORMAT = globals().get("ELEVEN_OUTPUT_FORMAT", "mp3_44100_128")

XTTS_LANGUAGE = globals().get("XTTS_LANGUAGE", "ar")
XTTS_MAX_REF_SEC_PER_SPEAKER = float(globals().get("XTTS_MAX_REF_SEC_PER_SPEAKER", 14.0))
XTTS_MIN_REF_SEC_PER_SPEAKER = float(globals().get("XTTS_MIN_REF_SEC_PER_SPEAKER", 2.0))
XTTS_MALE_FALLBACK_REF = Path(globals().get("XTTS_MALE_FALLBACK_REF", "/content/voices/male_ref.wav"))
XTTS_FEMALE_FALLBACK_REF = Path(globals().get("XTTS_FEMALE_FALLBACK_REF", "/content/voices/female_ref.wav"))

DEFAULT_FALLBACK_GENDER = globals().get("DEFAULT_FALLBACK_GENDER", "male")

if TTS_PROVIDER not in GENDER_VOICE_MAP:
    raise RuntimeError(f"TTS_PROVIDER={TTS_PROVIDER} not in voice map. Run Step 5.0 first.")

voices = GENDER_VOICE_MAP[TTS_PROVIDER]


def force_binary_gender(gender, default=DEFAULT_FALLBACK_GENDER):
    gender = str(gender or "").strip().lower()

    if gender == "female":
        return "female"

    if gender == "male":
        return "male"

    return default if default in ("male", "female") else "male"


gender_map = {
    spk: force_binary_gender(gender)
    for spk, gender in PIPELINE_PATHS.get("speaker_gender_map", {}).items()
}

# ──────── Inputs ────────
SRT_AR = Path(PIPELINE_PATHS.get(
    "srt_ar_dub_tts_numbers",
    PIPELINE_PATHS.get("srt_ar_dub", "")
))

if not SRT_AR.exists():
    raise FileNotFoundError(f"Arabic SRT not found: {SRT_AR}")

DIARIZATION_JSON = Path(PIPELINE_PATHS.get("diarization_json", ""))

if not DIARIZATION_JSON.exists():
    raise FileNotFoundError(f"Diarization JSON not found: {DIARIZATION_JSON}")

with open(DIARIZATION_JSON, "r", encoding="utf-8") as f:
    diar_data = json.load(f)

diar_segments = diar_data.get("segments", [])

if not diar_segments:
    raise RuntimeError("Diarization JSON has no segments.")

ref_audio = (
    PIPELINE_PATHS.get("speech_clean_16k")
    or PIPELINE_PATHS.get("audio_raw_wav")
    or PIPELINE_PATHS.get("audio_16k_mono")
)

if not ref_audio:
    raise RuntimeError("No reference audio found.")

REF_AUDIO = Path(ref_audio)

if not REF_AUDIO.exists():
    raise FileNotFoundError(f"Reference audio not found: {REF_AUDIO}")

# ──────── Output dirs ────────
tts_dir = DIRS["work"] / "05_tts"
cache_dir = tts_dir / "cache_tts"
ref_dir = tts_dir / "xtts_speaker_refs"

tts_dir.mkdir(parents=True, exist_ok=True)
cache_dir.mkdir(parents=True, exist_ok=True)
ref_dir.mkdir(parents=True, exist_ok=True)

video_stem = Path(PIPELINE_PATHS["input_video"]).stem
DUB_VOCALS_WAV = tts_dir / f"{video_stem}_dub_vocals_{target_sr}.wav"

# ──────── Provider deps ────────
if TTS_PROVIDER in ("elevenlabs", "elevenlabs_then_xtts"):
    try:
        from elevenlabs.client import ElevenLabs
    except ImportError:
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", "-U", "elevenlabs"
        ])
        from elevenlabs.client import ElevenLabs

if TTS_PROVIDER in ("xtts", "elevenlabs_then_xtts"):
    if "xtts_model" not in globals():
        try:
            from TTS.api import TTS
        except Exception:
            print("Installing XTTS / Coqui TTS for Python 3.12. This may take a few minutes...")

            subprocess.check_call([
                sys.executable, "-m", "pip", "install", "-q", "-U",
                "coqui-tts",
                "soundfile"
            ])

            from TTS.api import TTS

        import torch

        XTTS_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

        print(f"Loading XTTS v2 on {XTTS_DEVICE}...")
        xtts_model = TTS("tts_models/multilingual/multi-dataset/xtts_v2").to(XTTS_DEVICE)
        print("✅ XTTS v2 loaded.")

# ──────── Helpers ────────
def run(cmd):
    p = subprocess.run(cmd, capture_output=True, text=True)

    if p.returncode != 0:
        print("CMD:", " ".join(map(str, cmd)))
        print("STDERR:", (p.stderr or "")[-4000:])
        raise RuntimeError("Command failed")

    return p


def ms_from_pysrt_time(t):
    return (t.hours * 3600 + t.minutes * 60 + t.seconds) * 1000 + t.milliseconds


def clean_text_ar(s: str) -> str:
    s = (s or "").strip()
    s = re.sub(r"\s+", " ", s)

    for bad in ["[", "]", "♫", "♪"]:
        s = s.replace(bad, "")

    return s.strip()


def stable_hash(s: str) -> str:
    return hashlib.sha1(s.encode("utf-8")).hexdigest()[:16]


def atempo_chain(speed: float) -> str:
    if speed <= 0:
        speed = 1.0

    parts = []
    remain = float(speed)

    while remain > 2.0:
        parts.append(2.0)
        remain /= 2.0

    while remain < 0.5:
        parts.append(0.5)
        remain /= 0.5

    parts.append(remain)

    parts = [max(0.5, min(2.0, float(x))) for x in parts]

    return ",".join([f"atempo={x:.6f}" for x in parts])


def ensure_wav(in_audio: Path, out_wav: Path, sr: int = 24000):
    out_wav = Path(out_wav)
    out_wav.parent.mkdir(parents=True, exist_ok=True)

    run([
        "ffmpeg", "-y", "-loglevel", "error",
        "-i", str(in_audio),
        "-ac", "1",
        "-ar", str(sr),
        "-acodec", "pcm_s16le",
        str(out_wav)
    ])

    return out_wav


def find_speaker_for_time(start_ms: int, end_ms: int):
    best_speaker = None
    best_overlap = 0.0

    start_s = start_ms / 1000.0
    end_s = end_ms / 1000.0

    for seg in diar_segments:
        ov_start = max(float(seg["start"]), start_s)
        ov_end = min(float(seg["end"]), end_s)
        overlap = max(0.0, ov_end - ov_start)

        if overlap > best_overlap:
            best_overlap = overlap
            best_speaker = seg.get("speaker")

    return best_speaker


def is_elevenlabs_blocked_error(err):
    msg = str(err).lower()

    return (
        "detected_unusual_activity" in msg
        or "free tier usage disabled" in msg
        or "status_code: 401" in msg
        or "status_code=401" in msg
        or ("401" in msg and "eleven" in msg)
    )


# ──────── XTTS speaker references ────────
def build_xtts_speaker_refs():
    """
    Build one reference WAV per speaker from the original cleaned speech audio.
    This gives XTTS a better, less robotic voice than Edge TTS.
    """
    print("\n🎙️ Building XTTS speaker reference clips...")

    src = AudioSegment.from_file(str(REF_AUDIO))
    refs = {}

    speakers = sorted({
        seg.get("speaker")
        for seg in diar_segments
        if seg.get("speaker")
    })

    if not speakers:
        raise RuntimeError("No speakers found in diarization segments.")

    for spk in speakers:
        spk_segments = [
            seg for seg in diar_segments
            if seg.get("speaker") == spk
        ]

        spk_segments = sorted(
            spk_segments,
            key=lambda s: float(s.get("end", 0)) - float(s.get("start", 0)),
            reverse=True
        )

        combined = AudioSegment.silent(duration=0)
        total_ms = 0
        max_ms = int(XTTS_MAX_REF_SEC_PER_SPEAKER * 1000)

        for seg in spk_segments:
            start_ms = int(float(seg["start"]) * 1000)
            end_ms = int(float(seg["end"]) * 1000)

            dur_ms = max(0, end_ms - start_ms)

            if dur_ms < 700:
                continue

            clip = src[max(0, start_ms):max(0, end_ms)]

            if clip.dBFS < -45:
                continue

            combined += clip + AudioSegment.silent(duration=120)
            total_ms += len(clip)

            if total_ms >= max_ms:
                break

        out_ref = ref_dir / f"{spk}_xtts_ref.wav"

        if total_ms >= int(XTTS_MIN_REF_SEC_PER_SPEAKER * 1000):
            raw_ref = ref_dir / f"{spk}_xtts_ref_raw.wav"
            combined.export(str(raw_ref), format="wav")
            ensure_wav(raw_ref, out_ref, sr=24000)

            try:
                raw_ref.unlink()
            except Exception:
                pass

            refs[spk] = out_ref
            print(f"  ✅ {spk}: {total_ms/1000:.1f}s → {out_ref}")

        else:
            print(f"  ⚠️ {spk}: not enough clean reference audio ({total_ms/1000:.1f}s)")

    # Gender fallback reference files.
    for spk in speakers:
        if spk in refs:
            continue

        gender = force_binary_gender(gender_map.get(spk))
        fallback = XTTS_FEMALE_FALLBACK_REF if gender == "female" else XTTS_MALE_FALLBACK_REF

        if fallback.exists():
            out_ref = ref_dir / f"{spk}_xtts_gender_fallback.wav"
            ensure_wav(fallback, out_ref, sr=24000)
            refs[spk] = out_ref
            print(f"  ✅ {spk}: using gender fallback ref → {fallback}")

    # Last fallback: reuse any available speaker reference.
    if refs:
        first_ref = next(iter(refs.values()))

        for spk in speakers:
            if spk not in refs:
                refs[spk] = first_ref
                print(f"  ⚠️ {spk}: using shared fallback ref → {first_ref}")

    if not refs:
        raise RuntimeError(
            "XTTS could not build any speaker reference. "
            "Add /content/voices/male_ref.wav and /content/voices/female_ref.wav "
            "or check diarization audio."
        )

    return refs


XTTS_SPEAKER_REFS = {}

if TTS_PROVIDER in ("xtts", "elevenlabs_then_xtts"):
    XTTS_SPEAKER_REFS = build_xtts_speaker_refs()

# ──────── TTS functions ────────
def eleven_tts_to_mp3(text: str, voice_id: str, out_mp3: Path):
    key = os.getenv("ELEVEN_API_KEY", "").strip()

    if not key:
        raise RuntimeError("Missing ELEVEN_API_KEY")

    if not voice_id.strip():
        raise RuntimeError("Empty ElevenLabs voice_id")

    client = ElevenLabs(api_key=key)
    last_err = None

    for attempt in range(5):
        try:
            audio = client.text_to_speech.convert(
                text=text,
                voice_id=voice_id,
                model_id=ELEVEN_MODEL_ID,
                output_format=ELEVEN_OUTPUT_FORMAT,
            )

            data = audio if isinstance(audio, (bytes, bytearray)) else b"".join(audio)
            out_mp3.write_bytes(data)

            return out_mp3

        except Exception as e:
            last_err = e
            time.sleep(0.8 * (attempt + 1))

    raise RuntimeError(f"ElevenLabs TTS failed: {last_err}")


def xtts_tts_to_wav(text: str, speaker: str, out_wav: Path):
    if "xtts_model" not in globals():
        raise RuntimeError("XTTS model is not loaded. Run Step 5.0 first.")

    speaker_ref = XTTS_SPEAKER_REFS.get(speaker)

    if speaker_ref is None or not Path(speaker_ref).exists():
        if XTTS_SPEAKER_REFS:
            speaker_ref = next(iter(XTTS_SPEAKER_REFS.values()))
        else:
            raise RuntimeError("No XTTS speaker reference available.")

    out_wav = Path(out_wav)
    out_wav.parent.mkdir(parents=True, exist_ok=True)

    raw_wav = out_wav.with_suffix(".xtts_raw.wav")

    xtts_model.tts_to_file(
        text=text,
        speaker_wav=str(speaker_ref),
        language=XTTS_LANGUAGE,
        file_path=str(raw_wav),
    )

    ensure_wav(raw_wav, out_wav, sr=target_sr)

    try:
        raw_wav.unlink()
    except Exception:
        pass

    return out_wav


def tts_line_to_wav_resilient(text: str, voice: str, speaker: str, gender: str) -> tuple:
    """
    Returns:
      wav_path, provider_used, error_message
    """
    gender = force_binary_gender(gender)
    speaker = speaker or "SPEAKER_00"

    cache_payload = {
        "provider": TTS_PROVIDER,
        "text": text,
        "voice": voice,
        "speaker": speaker,
        "gender": gender,
        "eleven_model": ELEVEN_MODEL_ID,
        "eleven_format": ELEVEN_OUTPUT_FORMAT,
        "xtts_language": XTTS_LANGUAGE,
        "target_sr": target_sr,
    }

    key = stable_hash(json.dumps(cache_payload, ensure_ascii=False, sort_keys=True))

    wav_path = cache_dir / f"{key}_{target_sr}.wav"
    mp3_path = cache_dir / f"{key}.mp3"

    if wav_path.exists() and wav_path.stat().st_size > 0:
        return wav_path, "cache", None

    last_error = None

    # 1) ElevenLabs first.
    if TTS_PROVIDER in ("elevenlabs", "elevenlabs_then_xtts"):
        try:
            eleven_tts_to_mp3(text, voice, mp3_path)

            run([
                "ffmpeg", "-y", "-loglevel", "error",
                "-i", str(mp3_path),
                "-ac", "1",
                "-ar", str(target_sr),
                "-acodec", "pcm_s16le",
                str(wav_path)
            ])

            return wav_path, "elevenlabs", None

        except Exception as e:
            last_error = e

            if TTS_PROVIDER == "elevenlabs":
                raise

            print(f"  ⚠️ ElevenLabs failed. Switching to XTTS. Reason: {str(e)[:240]}")

    # 2) XTTS fallback / main provider.
    if TTS_PROVIDER in ("xtts", "elevenlabs_then_xtts"):
        try:
            xtts_tts_to_wav(
                text=text,
                speaker=speaker,
                out_wav=wav_path,
            )

            return wav_path, "xtts_v2", str(last_error) if last_error else None

        except Exception as e2:
            raise RuntimeError(
                "TTS failed.\n\n"
                f"ElevenLabs error:\n{last_error}\n\n"
                f"XTTS error:\n{e2}"
            )

    raise RuntimeError(f"Unsupported TTS_PROVIDER: {TTS_PROVIDER}")


def fit_to_slot(wav_in: Path, slot_ms: int, out_wav: Path):
    audio = AudioSegment.from_file(str(wav_in))
    dur = len(audio)

    if slot_ms <= 0:
        AudioSegment.silent(duration=0).export(str(out_wav), format="wav")
        return out_wav

    if dur <= 0:
        AudioSegment.silent(duration=slot_ms).export(str(out_wav), format="wav")
        return out_wav

    if dur > slot_ms:
        speed = dur / max(1, slot_ms)
        speed = min(float(speed), float(max_speedup))

        filt = atempo_chain(speed)

        run([
            "ffmpeg", "-y", "-loglevel", "error",
            "-i", str(wav_in),
            "-filter:a", f"{filt},aresample={target_sr}",
            "-ac", "1",
            "-ar", str(target_sr),
            str(out_wav)
        ])

        a2 = AudioSegment.from_file(str(out_wav))

        if len(a2) > slot_ms:
            a2 = a2[:slot_ms]
            a2.export(str(out_wav), format="wav")

        return out_wav

    pad = slot_ms - dur
    out = audio + AudioSegment.silent(duration=pad)
    out.export(str(out_wav), format="wav")

    return out_wav


# ──────── Build timeline ────────
ref_seg = AudioSegment.from_file(str(REF_AUDIO))
total_ms = len(ref_seg)

subs = pysrt.open(str(SRT_AR), encoding="utf-8")
timeline = AudioSegment.silent(duration=total_ms, frame_rate=target_sr)

print(f"🎬 Building dub timeline ({len(subs)} segments)...")
print(f"   Total duration: {total_ms/1000:.1f}s")

voice_usage = {}
provider_usage = {}
speaker_usage = {}

for s in subs:
    start_ms = ms_from_pysrt_time(s.start)
    end_ms = ms_from_pysrt_time(s.end)
    slot_ms = max(0, end_ms - start_ms)

    text = clean_text_ar(s.text)

    if not text:
        continue

    speaker = find_speaker_for_time(start_ms, end_ms) or "SPEAKER_00"
    gender = force_binary_gender(gender_map.get(speaker))

    voice = voices.get(gender, voices["male"])

    voice_usage[voice] = voice_usage.get(voice, 0) + 1
    speaker_usage[speaker] = speaker_usage.get(speaker, 0) + 1

    base_wav, provider_used, tts_error = tts_line_to_wav_resilient(
        text=text,
        voice=voice,
        speaker=speaker,
        gender=gender,
    )

    provider_usage[provider_used] = provider_usage.get(provider_used, 0) + 1

    meta_fit = json.dumps({
        "provider": provider_used,
        "speaker": speaker,
        "gender": gender,
        "voice": voice,
        "slot_ms": slot_ms,
        "text": text,
        "sr": target_sr,
        "max_speedup": max_speedup,
    }, ensure_ascii=False, sort_keys=True)

    fit_key = stable_hash(meta_fit)
    fitted = cache_dir / f"{fit_key}_fit.wav"

    if not fitted.exists() or fitted.stat().st_size == 0:
        fit_to_slot(base_wav, slot_ms, fitted)

    clip = AudioSegment.from_file(str(fitted))

    if fade_ms and fade_ms > 0 and len(clip) > (fade_ms * 2):
        clip = clip.fade_in(fade_ms).fade_out(fade_ms)

    if start_ms < total_ms:
        timeline = timeline.overlay(clip, position=max(0, start_ms))

    print(
        f"  {s.index:03d} | {speaker} | {gender} | "
        f"{provider_used} | {slot_ms/1000:.2f}s | {text[:60]}"
    )

timeline = timeline + float(vocal_gain_db)
timeline.export(str(DUB_VOCALS_WAV), format="wav")

PIPELINE_PATHS["dub_vocals_wav_48k"] = str(DUB_VOCALS_WAV)
PIPELINE_PATHS["tts_provider"] = TTS_PROVIDER
PIPELINE_PATHS["xtts_speaker_refs"] = {
    spk: str(path)
    for spk, path in XTTS_SPEAKER_REFS.items()
} if XTTS_SPEAKER_REFS else {}

print(f"\n✅ Dub vocals saved: {DUB_VOCALS_WAV}")

print("\n📊 Provider usage:")
for provider, n in sorted(provider_usage.items(), key=lambda x: -x[1]):
    print(f"  - {provider}: {n} segments")

print("\n📊 Speaker usage:")
for spk, n in sorted(speaker_usage.items(), key=lambda x: x[0]):
    print(f"  - {spk}: {n} segments")

print("\n📊 Voice usage:")
for voice, n in sorted(voice_usage.items(), key=lambda x: -x[1]):
    print(f"  - {voice}: {n} segments")

if XTTS_SPEAKER_REFS:
    print("\n🎙️ XTTS speaker references:")
    for spk, path in XTTS_SPEAKER_REFS.items():
        print(f"  - {spk}: {path}")


🎙️ Building XTTS speaker reference clips...
  ✅ SPEAKER_00: 9.1s → /content/dub_project/work/05_tts/xtts_speaker_refs/SPEAKER_00_xtts_ref.wav
  ✅ SPEAKER_01: 6.6s → /content/dub_project/work/05_tts/xtts_speaker_refs/SPEAKER_01_xtts_ref.wav
🎬 Building dub timeline (13 segments)...
   Total duration: 35.2s
  ⚠️ ElevenLabs failed. Switching to XTTS. Reason: ElevenLabs TTS failed: headers: {'date': 'Sat, 09 May 2026 13:31:21 GMT', 'server': 'uvicorn', 'content-length': '476', 'content-type': 'application/json', 'vary': 'Accept-Language', 'access-control-allow-origin': '*', 'access-control-allow
  001 | SPEAKER_00 | male | xtts_v2 | 4.21s | هل يمكنني مساعدتك؟
  ⚠️ ElevenLabs failed. Switching to XTTS. Reason: ElevenLabs TTS failed: headers: {'date': 'Sat, 09 May 2026 13:31:40 GMT', 'server': 'uvicorn', 'content-length': '476', 'content-type': 'application/json', 'vary': 'Accept-Language', 'access-control-allow-origin': '*', 'access-control-allow
  002 | SPEAKER_01 | female | xtts_v2 | 0.96

In [ ]:
#@title 6) Mix Background + Dub Vocals (Sidechain Ducking + Loudnorm) {display-mode:"form"}

music_gain_db = -2.0 #@param {type:"number"}
duck_threshold = 0.02 #@param {type:"number"}
duck_ratio = 8.0 #@param {type:"number"}
duck_attack_ms = 20 #@param {type:"integer"}
duck_release_ms = 250 #@param {type:"integer"}
target_sr = 48000 #@param {type:"integer"}

import subprocess
from pathlib import Path
from pydub import AudioSegment

def run(cmd):
    p = subprocess.run(cmd, capture_output=True, text=True)
    if p.returncode != 0:
        print("CMD:", " ".join(map(str, cmd)))
        print("STDERR:", (p.stderr or "")[-3000:])
        raise RuntimeError("Command failed")
    return p

# Preconditions
DUB_VOCALS = Path(PIPELINE_PATHS.get("dub_vocals_wav_48k", ""))
if not DUB_VOCALS.exists():
    raise FileNotFoundError("❌ Missing dub vocals. Run Step 5.1 first.")

mix_dir = DIRS["work"] / "06_mix"
mix_dir.mkdir(parents=True, exist_ok=True)

video_stem = Path(PIPELINE_PATHS["input_video"]).stem
BKG_PREP = mix_dir / f"{video_stem}_background_{target_sr}_stereo.wav"
VOC_PREP = mix_dir / f"{video_stem}_dubvoc_{target_sr}_stereo.wav"
FINAL_WAV = mix_dir / f"{video_stem}_final_mix_{target_sr}.wav"

#  background
bkg = PIPELINE_PATHS.get("background_wav")
bkg_path = Path(bkg) if bkg else None

if not bkg_path or not bkg_path.exists():
    demucs_root = Path(PIPELINE_PATHS.get("demucs_out_root", ""))
    bass = next(iter(sorted(demucs_root.glob("**/bass.wav"))), None) if demucs_root.exists() else None
    drums = next(iter(sorted(demucs_root.glob("**/drums.wav"))), None) if demucs_root.exists() else None
    other = next(iter(sorted(demucs_root.glob("**/other.wav"))), None) if demucs_root.exists() else None

    if bass and drums and other:
        tmp_bkg = mix_dir / f"{video_stem}_background_built.wav"
        run([
            "ffmpeg", "-y", "-loglevel", "error",
            "-i", str(bass), "-i", str(drums), "-i", str(other),
            "-filter_complex", "amix=inputs=3:duration=longest:dropout_transition=0",
            str(tmp_bkg)
        ])
        bkg_path = tmp_bkg
    else:
        raw = PIPELINE_PATHS.get("audio_raw_wav") or PIPELINE_PATHS.get("audio_16k_mono")
        if not raw:
            raise RuntimeError("❌ No background and no raw audio fallback found.")
        bkg_path = Path(raw)

print(f"🎵 Background source: {bkg_path}")

#  background → stereo + gain
run([
    "ffmpeg", "-y", "-loglevel", "error",
    "-i", str(bkg_path),
    "-filter:a", f"volume={music_gain_db}dB,aresample={target_sr},pan=stereo|c0=c0|c1=c0",
    str(BKG_PREP)
])

#  vocals → stereo
run([
    "ffmpeg", "-y", "-loglevel", "error",
    "-i", str(DUB_VOCALS),
    "-filter:a", f"aresample={target_sr},pan=stereo|c0=c0|c1=c0",
    str(VOC_PREP)
])

# المزج: ducking + loudnorm
fc = (
    f"[0:a][1:a]sidechaincompress="
    f"threshold={duck_threshold}:ratio={duck_ratio}:attack={duck_attack_ms}:release={duck_release_ms}"
    f"[ducked];"
    f"[ducked][1:a]amix=inputs=2:duration=longest:dropout_transition=0,"
    f"loudnorm=I=-16:TP=-1.5:LRA=11"
    f"[mix]"
)

run([
    "ffmpeg", "-y", "-loglevel", "error",
    "-i", str(BKG_PREP),
    "-i", str(VOC_PREP),
    "-filter_complex", fc,
    "-map", "[mix]",
    str(FINAL_WAV)
])

PIPELINE_PATHS["final_mix_wav"] = str(FINAL_WAV)
print(f"\n✅ Final mix saved: {FINAL_WAV}")

🎵 Background source: /content/dub_project/work/01_demucs/htdemucs/Learn English Romantic  Speak Fluently & Understand Real Conversations – FluentTalk Podcast - FluentTalk English Podcast (720p, h264)_raw/no_vocals.wav

✅ Final mix saved: /content/dub_project/work/06_mix/Learn English Romantic  Speak Fluently & Understand Real Conversations – FluentTalk Podcast - FluentTalk English Podcast (720p, h264)_final_mix_48000.wav


In [ ]:
#@title 7.0) Fix Arabic RTL in SRT (handle mixed EN/AR) {display-mode:"form"}

from pathlib import Path
import re
import pysrt

in_path = Path(PIPELINE_PATHS.get("srt_ar_dub", ""))
if not in_path.exists():
    raise FileNotFoundError(f"Missing srt_ar_dub: {in_path}")

out_path = in_path.with_name(in_path.stem + "_rtl_fixed.srt")

RLM = "\u200F"
LRM = "\u200E"
FSI = "\u2068"
PDI = "\u2069"

latin = re.compile(r"([A-Za-z0-9][A-Za-z0-9._:/@#\-+]*[A-Za-z0-9]?)")

def fix_line(t: str) -> str:
    t = (t or "").strip()
    if not t: return t
    t = re.sub(r"\s+", " ", t)
    t = latin.sub(lambda m: f"{FSI}{LRM}{m.group(1)}{PDI}", t)
    return RLM + t

subs = pysrt.open(str(in_path), encoding="utf-8")
for s in subs:
    lines = (s.text or "").splitlines()
    s.text = "\n".join(fix_line(x) for x in lines)

subs.save(str(out_path), encoding="utf-8")
PIPELINE_PATHS["srt_ar_dub_rtl_fixed"] = str(out_path)

print(f"✅ RTL-fixed SRT saved: {out_path}")

✅ RTL-fixed SRT saved: /content/dub_project/work/04_translate/Learn English Romantic  Speak Fluently & Understand Real Conversations – FluentTalk Podcast - FluentTalk English Podcast (720p, h264)_ar_dub_rtl_fixed.srt


In [32]:
#@title 7.1) Render Final Dubbed Video (MP4) {display-mode:"form"}

add_soft_subtitles = True #@param {type:"boolean"}
use_ar_srt = True #@param {type:"boolean"}

import subprocess
from pathlib import Path

def run(cmd):
    p = subprocess.run(cmd, capture_output=True, text=True)
    if p.returncode != 0:
        print("CMD:", " ".join(map(str, cmd)))
        print("STDERR:", (p.stderr or "")[-3000:])
        raise RuntimeError("Command failed")
    return p

# Locate video without audio
video_no_audio = Path(PIPELINE_PATHS.get("video_no_audio", ""))
if not video_no_audio.exists():
    cands = sorted((DIRS["extract"]).glob("*_no_audio.mp4"))
    video_no_audio = cands[-1] if cands else None
if not video_no_audio or not video_no_audio.exists():
    raise FileNotFoundError("❌ video_no_audio not found. Run Step 1.2 first.")

FINAL_WAV = Path(PIPELINE_PATHS.get("final_mix_wav", ""))
if not FINAL_WAV.exists():
    raise FileNotFoundError("❌ final_mix_wav not found. Run Step 6 first.")

out_dir = DIRS["output"]
out_dir.mkdir(parents=True, exist_ok=True)

video_stem = Path(PIPELINE_PATHS["input_video"]).stem
OUT_MP4 = out_dir / f"{video_stem}_dubbed.mp4"

cmd = [
    "ffmpeg", "-y",
    "-i", str(video_no_audio),
    "-i", str(FINAL_WAV),
]
maps = ["-map", "0:v:0", "-map", "1:a:0"]

if add_soft_subtitles:
    if use_ar_srt:
        srt_path = Path(PIPELINE_PATHS.get("srt_ar_dub_rtl_fixed",
                                            PIPELINE_PATHS.get("srt_ar_dub", "")))
    else:
        srt_path = Path(PIPELINE_PATHS.get("asr_srt_en", ""))

    if srt_path and srt_path.exists():
        cmd += ["-i", str(srt_path)]
        maps += ["-map", "2:s:0"]
    else:
        print("⚠️  Subtitle file not found; continuing without subtitles.")
        add_soft_subtitles = False

cmd += maps
cmd += ["-c:v", "copy", "-c:a", "aac", "-b:a", "192k"]

if add_soft_subtitles:
    cmd += ["-c:s", "mov_text"]

cmd += ["-shortest", str(OUT_MP4)]

run(cmd)

PIPELINE_PATHS["final_video_mp4"] = str(OUT_MP4)
print(f"\n🎉 Final dubbed video: {OUT_MP4}")
print(f"   Size: {OUT_MP4.stat().st_size / (1024*1024):.1f} MB")


🎉 Final dubbed video: /content/dub_project/output/Learn English Romantic  Speak Fluently & Understand Real Conversations – FluentTalk Podcast - FluentTalk English Podcast (720p, h264)_dubbed.mp4
   Size: 4.6 MB


In [33]:
#@title 7.2) Save to Drive + Download {display-mode:"form"}

copy_to_drive = False #@param {type:"boolean"}
auto_download = True #@param {type:"boolean"}

from pathlib import Path
import shutil

final_video = Path(PIPELINE_PATHS.get("final_video_mp4", ""))
final_mix   = Path(PIPELINE_PATHS.get("final_mix_wav", ""))
ar_srt      = Path(PIPELINE_PATHS.get("srt_ar_dub_rtl_fixed",
                                       PIPELINE_PATHS.get("srt_ar_dub", "")))
diar_json   = Path(PIPELINE_PATHS.get("diarization_json", ""))

print("📦 Pipeline artifacts:")
print(f"  🎬 Final video    : {final_video} {'✅' if final_video.exists() else '❌'}")
print(f"  🎵 Final mix WAV  : {final_mix}   {'✅' if final_mix.exists() else '❌'}")
print(f"  📝 Arabic SRT     : {ar_srt}      {'✅' if ar_srt.exists() else '❌'}")
print(f"  📊 Diarization JSON: {diar_json}  {'✅' if diar_json.exists() else '❌'}")

if copy_to_drive:
    try:
        drive_upload_path
    except NameError:
        drive_upload_path = Path("/content/gdrive/MyDrive") / "dub_project"
        drive_upload_path.mkdir(parents=True, exist_ok=True)

    print("\n📤 Copying to Drive...")
    for f in [final_video, final_mix, ar_srt, diar_json]:
        if f.exists():
            dest = drive_upload_path / f.name
            shutil.copy(str(f), str(dest))
            print(f"  ✅ {dest}")

if auto_download:
    print("\n⬇️  Downloading...")
    from google.colab import files
    if final_video.exists(): files.download(str(final_video))
    if ar_srt.exists():      files.download(str(ar_srt))
    if diar_json.exists():   files.download(str(diar_json))

print("\n✅ Step 7 done. ")
print("\n💡 Optional: Run Step 8 for Lip-Sync (MuseTalk + CodeFormer) for higher quality.")

📦 Pipeline artifacts:
  🎬 Final video    : /content/dub_project/output/Learn English Romantic  Speak Fluently & Understand Real Conversations – FluentTalk Podcast - FluentTalk English Podcast (720p, h264)_dubbed.mp4 ✅
  🎵 Final mix WAV  : /content/dub_project/work/06_mix/Learn English Romantic  Speak Fluently & Understand Real Conversations – FluentTalk Podcast - FluentTalk English Podcast (720p, h264)_final_mix_48000.wav   ✅
  📝 Arabic SRT     : /content/dub_project/work/04_translate/Learn English Romantic  Speak Fluently & Understand Real Conversations – FluentTalk Podcast - FluentTalk English Podcast (720p, h264)_ar_dub_rtl_fixed.srt      ✅
  📊 Diarization JSON: /content/dub_project/work/03_asr_diarize/Learn English Romantic  Speak Fluently & Understand Real Conversations – FluentTalk Podcast - FluentTalk English Podcast (720p, h264)_diarization.json  ✅

⬇️  Downloading...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅ Step 7 done. 

💡 Optional: Run Step 8 for Lip-Sync (MuseTalk + CodeFormer) for higher quality.


In [ ]:
#@title 8.0) Save Pipeline State (قبل إعادة تشغيل runtime) {display-mode:"form"}


import json
from pathlib import Path

run_lipsync = True #@param {type:"boolean"}

if not run_lipsync:
    print("⏸️  Lip-sync skipped. Pipeline complete at Step 7.")
    print("    Final video: ", PIPELINE_PATHS.get("final_video_mp4"))
else:
    state_file = Path("/content/dub_project/pipeline_state.json")
    state_file.parent.mkdir(parents=True, exist_ok=True)

    serializable = {}
    for k, v in PIPELINE_PATHS.items():
        if v is None:
            serializable[k] = None
        else:
            serializable[k] = str(v)

    with open(state_file, "w", encoding="utf-8") as f:
        json.dump(serializable, f, ensure_ascii=False, indent=2)

    print(f"💾 Pipeline state saved: {state_file}")
    print("\n⚠️  Lip-sync requires Python 3.10 environment.")
    print("    Running the next cell will REINSTALL everything and reset runtime.")
    print("    Make sure you have:")
    print("    - Final video from Step 7 ✅")
    print("    - Final mix WAV from Step 6 ✅")
    print("    - GPU enabled ✅")

💾 Pipeline state saved: /content/dub_project/pipeline_state.json

⚠️  Lip-sync requires Python 3.10 environment.
    Running the next cell will REINSTALL everything and reset runtime.
    Make sure you have:
    - Final video from Step 7 ✅
    - Final mix WAV from Step 6 ✅
    - GPU enabled ✅


In [ ]:
#@title 8.1) Install Python 3.10 + MuseTalk environment {display-mode:"form"}


!wget -qO miniconda.sh https://repo.anaconda.com/miniconda/Miniconda3-py310_23.5.2-0-Linux-x86_64.sh

!bash ./miniconda.sh -b -f -p /usr/local

import sys
sys.path.append('/usr/local/lib/python3.10/site-packages/')

print("✅ Python 3.10 installed.")
print("⏭️  Run next cell (8.2) to clone MuseTalk.")

PREFIX=/usr/local
Unpacking payload ...
                                                                                   
Installing base environment...





Preparing transaction: - \ | / - done
Executing transaction: | / - \ | / - \ | / - \ | / - \ | / - \ | / - \ | done
installation finished.
    You currently have a PYTHONPATH environment variable set. This may cause
    unexpected behavior when running the Python interpreter in Miniconda3.
    For best results, please verify that your PYTHONPATH only points to
    directories of packages that are compatible with the Python interpreter
    in Miniconda3: /usr/local
✅ Python 3.10 installed.
⏭️  Run next cell (8.2) to clone MuseTalk.


In [ ]:
#@title 8.2) Clone MuseTalk + Install Requirements {display-mode:"form"}

# Clone MuseTalk
!git clone https://github.com/TMElyralab/MuseTalk.git
%cd MuseTalk

!pip install -r requirements.txt

!pip install "huggingface-hub>=0.19.3,<1.0" --force-reinstall

print("✅ MuseTalk cloned and base requirements installed.")

Cloning into 'MuseTalk'...
remote: Enumerating objects: 534, done.
remote: Counting objects: 100% (131/131), done.
remote: Compressing objects: 100% (68/68), done.
remote: Total 534 (delta 83), reused 63 (delta 63), pack-reused 403 (from 1)
Receiving objects: 100% (534/534), 25.75 MiB | 41.91 MiB/s, done.
Resolving deltas: 100% (219/219), done.
/content/MuseTalk
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 14.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 290.1/290.1 kB 22.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.1/17.1 MB 49.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 585.9/585.9 MB 707.4 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 115.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.2/62.2 MB 11.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 76.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 72.9 MB/s eta 0:

In [ ]:
#@title 8.3) Install PyTorch 2.0.1 + mmcv stack {display-mode:"form"}

!pip uninstall -y torch torchvision torchaudio

!pip install torch==2.0.1 torchvision==0.15.2 torchaudio==2.0.2 --index-url https://download.pytorch.org/whl/cu118

!pip install -U openmim
!mim install mmengine
!mim install "mmcv>=2.0.1" -f https://download.openmmlab.com/mmcv/dist/cu118/torch2.0.0/index.html
!mim install "mmdet>=3.1.0"
!mim install "mmpose>=1.1.0"

print("✅ PyTorch 2.0.1 + mmcv stack installed.")

In [ ]:
#@title 8.4) Download MuseTalk Model Weights {display-mode:"form"}

import os
import json

if not os.getcwd().endswith("MuseTalk"):
    %cd /content/MuseTalk

print("🛠️ جاري تحميل أوزان النماذج وإعداد VAE...")

# 1. إعداد المجلدات
directories = [
    "./models/sd-vae", "./models/whisper", "./models/dwpose",
    "./models/musetalk", "./models/musetalkV15", "./models/face-parse-bisent"
]
for d in directories:
    os.makedirs(d, exist_ok=True)

print("📥 VAE weights & config...")
os.system("wget -qO ./models/sd-vae/config.json https://huggingface.co/stabilityai/sd-vae-ft-mse/resolve/main/config.json")
os.system("wget -qO ./models/sd-vae/diffusion_pytorch_model.bin https://huggingface.co/stabilityai/sd-vae-ft-mse/resolve/main/diffusion_pytorch_model.bin")

# 3. UNet, DWPose, Face Parsing
print("📥 UNet, DWPose, Face Parsing...")
if not os.path.exists("./models/musetalkV15/unet.pth"):
    os.system("wget -qO ./models/musetalkV15/unet.pth https://huggingface.co/TMElyralab/MuseTalk/resolve/main/musetalk/pytorch_model.bin")
    os.system("cp ./models/musetalkV15/unet.pth ./models/musetalk/unet.pth")

os.system("wget -qO ./models/dwpose/dw-ll_ucoco_384.pth https://huggingface.co/yzd-v/DWPose/resolve/main/dw-ll_ucoco_384.pth")
os.system("wget -qO ./models/face-parse-bisent/resnet18-5c106cde.pth https://download.pytorch.org/models/resnet18-5c106cde.pth")

# Face parsing model
if not os.path.exists("./models/face-parse-bisent/79999_iter.pth") or os.path.getsize("./models/face-parse-bisent/79999_iter.pth") < 50000000:
    !gdown 154JgKpzCPW82qINcVieuPH3fZ2e0P812 -O ./models/face-parse-bisent/79999_iter.pth

# 4. Whisper config
os.system("wget -qO ./models/musetalk/config.json https://huggingface.co/TMElyralab/MuseTalk/resolve/main/musetalk/musetalk.json")
os.system("cp ./models/musetalk/config.json ./models/musetalkV15/config.json")

whisper_preprocessor = {
    "feature_extractor_type": "WhisperFeatureExtractor",
    "feature_size": 80, "hop_length": 160, "n_fft": 400,
    "n_samples": 480000, "nb_max_frames": 3000,
    "padding_value": 0.0, "return_attention_mask": False,
    "sampling_rate": 16000
}
with open('./models/whisper/preprocessor_config.json', 'w') as f:
    json.dump(whisper_preprocessor, f)

for file in ["config.json", "tokenizer_config.json", "vocab.json", "merges.txt", "pytorch_model.bin"]:
    os.system(f"wget -qO ./models/whisper/{file} https://huggingface.co/openai/whisper-tiny/resolve/main/{file}")

print("\n✅ كل الأوزان والإعدادات جاهزة.")

🛠️ جاري تحميل أوزان النماذج وإعداد VAE...
📥 VAE weights & config...
📥 UNet, DWPose, Face Parsing...
Downloading...
From: https://drive.google.com/uc?id=154JgKpzCPW82qINcVieuPH3fZ2e0P812
To: /content/MuseTalk/models/face-parse-bisent/79999_iter.pth
100% 53.3M/53.3M [00:00<00:00, 133MB/s]

✅ كل الأوزان والإعدادات جاهزة.


In [ ]:
#@title 8.5) Prepare MuseTalk config (uses Step 7 output) {display-mode:"form"}

import json, os, shutil, subprocess
from pathlib import Path

state_file = Path("/content/dub_project/pipeline_state.json")
if not state_file.exists():
    raise FileNotFoundError("❌ Pipeline state not found. Run Step 8.0 first.")

with open(state_file, "r", encoding="utf-8") as f:
    PIPELINE_PATHS = json.load(f)

final_video_path = PIPELINE_PATHS.get("final_video_mp4")
final_mix_path   = PIPELINE_PATHS.get("final_mix_wav")

if not final_video_path or not Path(final_video_path).exists():
    raise FileNotFoundError(f"❌ Final video not found: {final_video_path}")
if not final_mix_path or not Path(final_mix_path).exists():
    raise FileNotFoundError(f"❌ Final mix not found: {final_mix_path}")

print(f"🎬 Input video: {final_video_path}")
print(f"🎵 Input audio: {final_mix_path}")

musetalk_video = "/content/video.mp4"
musetalk_audio = "/content/audio.wav"

shutil.copy(final_video_path, musetalk_video)

subprocess.run([
    "ffmpeg", "-y", "-loglevel", "error",
    "-i", final_mix_path,
    "-acodec", "pcm_s16le",
    "-ar", "16000",
    "-ac", "1",
    musetalk_audio
], check=True)

print(f"✅ Copied to MuseTalk paths:")
print(f"  - {musetalk_video}")
print(f"  - {musetalk_audio}")

config_data = {
    "dub_task": {
        "video_path": musetalk_video,
        "audio_path": musetalk_audio,
        "bbox_shift": 0
    }
}

os.makedirs("/content/MuseTalk", exist_ok=True)
config_path = "/content/MuseTalk/dub_config.json"
with open(config_path, "w") as f:
    json.dump(config_data, f, indent=4)

print(f"\n✅ MuseTalk config created: {config_path}")

FileNotFoundError: ❌ Final video not found: None

In [ ]:
#@title 8.6) Run MuseTalk (Lip-Sync) with smoothing patch {display-mode:"form"}


import os

os.system("cd /content/MuseTalk && git checkout scripts/inference.py")

inference_path = "/content/MuseTalk/scripts/inference.py"
config_path = "/content/MuseTalk/dub_config.json"

smoothing_logic = '''
def smooth_coordinates_robust(coord_list, window_size=15):
    import numpy as np
    smoothed = []
    half = window_size // 2
    for i in range(len(coord_list)):
        start = max(0, i - half)
        end = min(len(coord_list), i + half + 1)
        window = coord_list[start:end]

        # تنعيم كل إحداثي على حدة (للحفاظ على مرونة الحجم)
        y1 = int(np.median([c[0] for c in window]))
        y2 = int(np.median([c[1] for c in window]))
        x1 = int(np.median([c[2] for c in window]))
        x2 = int(np.median([c[3] for c in window]))

        smoothed.append((y1, y2, x1, x2))
    return smoothed
'''

# 3. حقن الكود
with open(inference_path, "r", encoding="utf-8") as f:
    lines = f.readlines()

with open(inference_path, "w", encoding="utf-8") as f:
    f.write(smoothing_logic + "\n")
    for line in lines:
        f.write(line)
        if "coord_list, frame_list = get_landmark_and_bbox(input_img_list, bbox_shift)" in line:
            indent = line[:len(line) - len(line.lstrip())]
            f.write(f"{indent}coord_list = smooth_coordinates_robust(coord_list)\n")
            f.write(f"{indent}print('🛡️ Smoothing window=15 ACTIVE')\n")

print("✅ Smoothing patch injected.")

# 4. الرندرة
os.environ['MPLBACKEND'] = 'Agg'
print("\n🎬 Running MuseTalk inference (this may take 10-30 minutes)...")

!cd /content/MuseTalk && MPLBACKEND=Agg python -m scripts.inference \
    --inference_config "{config_path}" \
    --bbox_shift 0 \
    --output_vid_name "MuseTalk_output.mp4"

print("\n✅ MuseTalk inference complete.")

In [ ]:
#@title 8.7) Install CodeFormer + dependencies {display-mode:"form"}

import os
import sys
import site
import subprocess
from pathlib import Path

def sh(cmd, cwd=None):
    print(">>", " ".join(cmd) if isinstance(cmd, list) else cmd)
    subprocess.check_call(cmd, cwd=cwd)

sh([sys.executable, "-m", "pip", "install", "-q", "numpy<2"])

# 2) Clone CodeFormer
os.chdir("/content")

if not Path("/content/CodeFormer").exists():
    sh(["git", "clone", "https://github.com/sczhou/CodeFormer.git"])

os.chdir("/content/CodeFormer")

sh([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])
sh([
    sys.executable, "-m", "pip", "install", "-q",
    "basicsr", "facexlib", "lpips", "realesrgan"
])


candidate_roots = []

try:
    candidate_roots.extend(site.getsitepackages())
except Exception:
    pass

try:
    candidate_roots.append(site.getusersitepackages())
except Exception:
    pass

candidate_roots.extend([
    f"/usr/local/lib/python{sys.version_info.major}.{sys.version_info.minor}/site-packages",
    f"/usr/lib/python{sys.version_info.major}.{sys.version_info.minor}/site-packages",
])

basicsr_path = None

for root in candidate_roots:
    root = Path(root)
    p = root / "basicsr" / "data" / "degradations.py"
    if p.exists():
        basicsr_path = p
        break

if basicsr_path is None:
    matches = list(Path("/usr/local/lib").glob("python*/site-packages/basicsr/data/degradations.py"))
    if matches:
        basicsr_path = matches[0]

if basicsr_path and basicsr_path.exists():
    content = basicsr_path.read_text(encoding="utf-8")

    old = "from torchvision.transforms.functional_tensor import rgb_to_grayscale"
    new = "from torchvision.transforms.functional import rgb_to_grayscale"

    if old in content:
        basicsr_path.write_text(content.replace(old, new), encoding="utf-8")
        print(f"✅ basicsr patched: {basicsr_path}")
    else:
        print(f"✅ basicsr already compatible: {basicsr_path}")
else:
    print("⚠️ basicsr/data/degradations.py not found. Patch skipped.")

sh([sys.executable, "scripts/download_pretrained_models.py", "facelib"], cwd="/content/CodeFormer")
sh([sys.executable, "scripts/download_pretrained_models.py", "CodeFormer"], cwd="/content/CodeFormer")

print("\n✅ CodeFormer جاهز.")

In [ ]:
#@title 8.8) Run CodeFormer Face Enhancement {display-mode:"form"}

# ────────  CodeFormer ────────
fidelity_weight = 0.95 #@param {type:"number"}
upscale_factor = 1     #@param {type:"integer"}
use_face_upsample = True #@param {type:"boolean"}
use_bg_upsampler = False #@param {type:"boolean"}

import os, json, shutil, subprocess, time
from pathlib import Path


def find_latest_musetalk_video():
    search_roots = [
        Path("/content/MuseTalk/results"),
        Path("/content/MuseTalk/result"),
        Path("/content/MuseTalk"),
        Path("/content/dub_project/output"),
        Path("/content/dub_project/work"),
    ]

    exclude_keywords = [
        "pretrain",
        "pretrained",
        "checkpoint",
        "checkpoints",
        "assets",
        "examples",
        "input",
        "source",
        "audio",
        "tmp",
        "temp",
        "__pycache__",
    ]

    candidates = []

    if "PIPELINE_PATHS" in globals():
        for key in [
            "musetalk_video",
            "musetalk_output",
            "lipsync_video",
            "final_musetalk_video",
        ]:
            val = PIPELINE_PATHS.get(key)
            if val and Path(val).exists() and Path(val).suffix.lower() == ".mp4":
                candidates.append(Path(val))

    for root in search_roots:
        if not root.exists():
            continue

        for p in root.rglob("*.mp4"):
            p_str = str(p).lower()

            if any(k in p_str for k in exclude_keywords):
                continue

            try:
                size_mb = p.stat().st_size / (1024 * 1024)
                mtime = p.stat().st_mtime
            except Exception:
                continue

            if size_mb < 0.2:
                continue

            candidates.append(p)

    candidates = list(dict.fromkeys(candidates))

    if not candidates:
        print("❌ لم أجد أي فيديو mp4 ناتج من MuseTalk.")
        print("\n🔍 Debug: آخر ملفات mp4 الموجودة داخل /content:")

        debug_files = sorted(
            Path("/content").rglob("*.mp4"),
            key=lambda x: x.stat().st_mtime if x.exists() else 0,
            reverse=True
        )[:30]

        for p in debug_files:
            try:
                print(
                    f" - {p} | "
                    f"{p.stat().st_size / (1024 * 1024):.2f} MB | "
                    f"{time.ctime(p.stat().st_mtime)}"
                )
            except Exception:
                print(f" - {p}")

        raise FileNotFoundError(
            "❌ MuseTalk output not found. "
            "راجع خلية MuseTalk السابقة لأنها غالباً لم تنتج الفيديو."
        )

    candidates = sorted(
        candidates,
        key=lambda p: p.stat().st_mtime,
        reverse=True
    )

    print("🎬 MuseTalk video candidates:")
    for i, p in enumerate(candidates[:10], 1):
        print(
            f"  {i}. {p} | "
            f"{p.stat().st_size / (1024 * 1024):.2f} MB | "
            f"{time.ctime(p.stat().st_mtime)}"
        )

    return candidates[0]


musetalk_video = find_latest_musetalk_video()
print(f"\n✅ Selected MuseTalk output: {musetalk_video}")

%cd /content/CodeFormer
frames_dir = Path("/content/CodeFormer/musetalk_frames")
frames_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
#@title Save / Copy Final Lip-sync Video Safely { display-mode: "form" }

from pathlib import Path
import shutil, time, os

search_roots = [
    Path("/content/MuseTalk/results"),
    Path("/content/MuseTalk/result"),
    Path("/content/MuseTalk"),
    Path("/content/CodeFormer/results"),
    Path("/content/dub_project/output"),
    Path("/content/dub_project/work"),
    Path("/content"),
]

def is_valid_video(p: Path):
    if not p:
        return False
    if not isinstance(p, Path):
        p = Path(p)
    if not p.exists():
        return False
    if not p.is_file():
        return False
    if p.suffix.lower() not in [".mp4", ".mov", ".mkv", ".avi"]:
        return False
    if p.stat().st_size < 1024 * 100:  
        return False
    return True

def find_latest_video():
    candidates = []

    if "PIPELINE_PATHS" in globals():
        possible_keys = [
            "final_lipsync_video",
            "enhanced_lipsync_video",
            "codeformer_video",
            "musetalk_video",
            "musetalk_output",
            "lipsync_video",
            "final_video",
        ]

        for key in possible_keys:
            val = PIPELINE_PATHS.get(key)
            if val:
                p = Path(val)
                if is_valid_video(p):
                    candidates.append(p)

    exclude_keywords = [
        "input",
        "source",
        "audio",
        "tmp",
        "temp",
        "pretrain",
        "pretrained",
        "checkpoint",
        "checkpoints",
        "assets",
        "__pycache__",
    ]

    for root in search_roots:
        if not root.exists():
            continue

        for p in root.rglob("*.mp4"):
            p_str = str(p).lower()

            if any(k in p_str for k in exclude_keywords):
                continue

            if is_valid_video(p):
                candidates.append(p)

    candidates = list(dict.fromkeys(candidates))

    if not candidates:
        print("❌ لم أجد أي ملف فيديو نهائي صالح.")
        print("\n🔍 آخر ملفات MP4 الموجودة داخل /content:")

        debug_files = sorted(
            Path("/content").rglob("*.mp4"),
            key=lambda x: x.stat().st_mtime if x.exists() else 0,
            reverse=True
        )[:30]

        for p in debug_files:
            try:
                print(
                    f" - {p} | "
                    f"{p.stat().st_size / (1024 * 1024):.2f} MB | "
                    f"{time.ctime(p.stat().st_mtime)} | "
                    f"is_file={p.is_file()}"
                )
            except Exception:
                print(f" - {p}")

        raise FileNotFoundError(
            "❌ Final lip-sync video not found. "
            "المتغير final_lipsync كان يشير إلى مجلد وليس ملف فيديو."
        )

    candidates = sorted(
        candidates,
        key=lambda p: p.stat().st_mtime,
        reverse=True
    )

    print("🎬 Video candidates:")
    for i, p in enumerate(candidates[:10], 1):
        print(
            f"  {i}. {p} | "
            f"{p.stat().st_size / (1024 * 1024):.2f} MB | "
            f"{time.ctime(p.stat().st_mtime)}"
        )

    return candidates[0]


final_lipsync = find_latest_video()

print(f"\n🎬 Final lip-sync video: {final_lipsync}")
print(f"   Size: {final_lipsync.stat().st_size / (1024 * 1024):.2f} MB")

if "PIPELINE_PATHS" in globals():
    PIPELINE_PATHS["final_lipsync_video"] = str(final_lipsync)

drive_target = Path("/content/drive/MyDrive/dub_project_outputs")

if Path("/content/drive/MyDrive").exists():
    drive_target.mkdir(parents=True, exist_ok=True)

    dest = drive_target / final_lipsync.name

    if dest.resolve() != final_lipsync.resolve():
        shutil.copy2(str(final_lipsync), str(dest))
        print(f"✅ Copied to Drive: {dest}")
    else:
        print(f"✅ Already in Drive: {dest}")
else:
    print("⚠️ Google Drive غير مركّب. سيتم الاحتفاظ بالفيديو داخل Colab فقط.")

print("\n✅ Done.")